# Step 4 — merge blocks into final city segments

**# of cells in notebook:** 2 code cells

## Purpose

Create the final city-segment geography by merging blocks within each `zones_5` zone. The workflow uses population, rook adjacency, building-density similarity, compactness, and future merge feasibility to create segments whose populations are generally targeted toward the 500–1,000 range. A second stage then resolves disconnected under-500 island segments by attaching them to nearby same-zone segments where appropriate.

Open-space and airport blocks are excluded from merging and remain standalone segments throughout both stages.

## Input

### Cell 1 input

- GeoPackage: `E:\_johannesburg\_analysis\segments_v2\blocks_5_with_building_stats.gpkg`
- Layer: `blocks_5_with_building_stats`

Important fields include:

- `block_id`
- `zones_5_ID`
- `zones_5_pop`
- `population`
- `open_space`
- `airport`
- `block_area_m2`
- `bldg_count`
- `bldg_area_sum`

The topology diagnostics from Step 3 inform the coordinate-precision setting used by the merge. The current code uses a `0.01 m` working precision grid.

### Cell 2 input

Cell 2 uses the stage-1 outputs created by Cell 1:

- `blocks_5_citywide_merged.gpkg`
- `segments_best`
- `blocks_with_segment_id`

## Output

### Stage 1

**Folder**
- `E:\_johannesburg\_analysis\segments_v2\citywide_merge_experiment`

**GeoPackage**
- `blocks_5_citywide_merged.gpkg`

**Primary layers**
- `segments_best`
- `blocks_with_segment_id`

Supporting outputs include:

- `city_summary.csv`
- `zone_summary.csv`
- `component_summary.csv`
- `run_summary.csv`
- `merge_log_best.csv`
- `segment_summary.csv`
- `geometry_processing_summary.csv`

### Final stage

**Folder**
- `E:\_johannesburg\_analysis\segments_v2\citywide_island_resolution`

**GeoPackage**
- `blocks_5_citywide_islands_resolved.gpkg`

**Primary layers**
- `segments_islands_resolved`
- `blocks_with_final_segment_id`
- `island_attachment_lines`

Supporting outputs include:

- `city_island_summary.csv`
- `zone_island_summary.csv`
- `island_run_summary.csv`
- `island_merge_log_best.csv`
- `island_candidate_audit_best.csv`
- `stage1_to_final_crosswalk.csv`
- `final_segment_summary.csv`

## Main logic

### Cell 1 — create first-stage citywide segments

1. Read `blocks_5_with_building_stats` and validate the required attributes and polygon geometry.
2. Apply the selected coordinate precision in memory. With the current setting, coordinates are processed on a `0.01 m` grid.
3. Preserve the source file; precision-cleaned geometry is used only for the working and output layers.
4. Identify merge-excluded blocks:
   - `open_space = 1`;
   - `airport = 1`.
5. Keep each merge-excluded block as its own standalone segment.

#### Zones with population of 1,000 or less

6. For a `zones_5` zone with `zones_5_pop <= 1,000`, dissolve all merge-eligible blocks in the zone directly into one segment.
7. Disconnected pieces are allowed, so this output may be multipart.
8. Open-space and airport blocks remain separate and are not included in this dissolve.

#### Zones with population greater than 1,000

9. Build a strict same-zone rook-adjacency graph among merge-eligible blocks.
10. Separate the graph into connected components.
11. Retain degree-zero blocks as standalone `island_deferred` segments for later processing.
12. Process each non-island connected component independently.

#### Iterative merging

13. Begin with individual blocks as candidate regions.
14. Iteratively evaluate adjacent regions for merging.
15. Use a target population range of approximately 500–1,000 people:
    - population below 500 is treated as underfilled;
    - population above 1,000 is allowed when necessary but penalized.
16. Evaluate candidate merges using several considerations, including:
    - whether a merge would strand neighboring under-500 regions;
    - similarity in building-area density and building-count density;
    - compactness of the resulting polygon;
    - resulting population;
    - narrow shared-boundary or “neck” effects;
    - population overflow above 1,000.
17. Treat differences of 20% or less in the two building-density measures as practically equivalent and apply no building-heterogeneity penalty within that range.
18. Perform 20 seeded near-greedy runs for each iterative connected component.
19. Select the best run according to the resulting population, unresolved-region, overflow, geometry, and heterogeneity outcomes.
20. Assign stable citywide segment IDs and dissolve the source blocks by the selected segment assignment.
21. Verify preservation of population both citywide and within each zone.
22. Write the stage-1 segment, block-crosswalk, run-summary, merge-log, and QA outputs.

Stage-1 segment types distinguish ordinary iterative segments, unresolved under-500 components, deferred islands, small-zone segments, and merge-excluded open-space/airport segments.

### Cell 2 — resolve disconnected under-500 island segments

1. Read `segments_best` and `blocks_with_segment_id` from Cell 1.
2. Leave zones with `zones_5_pop <= 1,000` unchanged.
3. Keep all open-space and airport-derived segments unchanged; they can neither initiate nor receive stage-2 attachment merges.
4. Identify under-500 segments originating from the following stage-1 types:
   - `island_deferred`;
   - `unavoidable_under500_component`;
   - `iterative_under500_unresolved`.
5. Treat those under-500 island-derived segments as stage-2 merge targets.
6. For each target, consider up to the eight nearest current same-zone eligible regions using polygon-to-polygon distance.
7. Prefer candidates that keep the resulting population at or below 1,000.
8. If none of the considered candidates can remain at or below 1,000, allow an over-1,000 merge but penalize the overflow.
9. Score candidate attachments using:
   - future feasibility for other unresolved islands;
   - polygon distance;
   - building-density heterogeneity;
   - resulting population;
   - overflow above 1,000.
10. Perform 20 seeded near-greedy runs for each zone containing stage-2 targets.
11. Select the best run by prioritizing:
    - the fewest unresolved under-500 island segments;
    - the smallest remaining population deficit;
    - reduced population overflow;
    - shorter attachment distances;
    - lower building heterogeneity.
12. Dissolve the selected stage-1 segments into final city segments.
13. Assign final segment IDs to the source blocks.
14. Create `island_attachment_lines` showing the spatial attachments made during stage 2.
15. Create a crosswalk between stage-1 and final segment IDs.
16. Verify that population is preserved.
17. Write the final segment layers and QA/diagnostic tables.

The principal final city-segment layer produced by this workflow is `segments_islands_resolved`.


In [ ]:
"""
Citywide constrained block merging for Johannesburg.

The workflow is applied independently within each zones_5_ID.

Rules
-----
1. open_space == 1:
   - excluded from all adjacency graphs and merges;
   - retained as its own standalone segment.

2. zones_5_pop <= 1,000:
   - all non-open-space blocks in that zone are dissolved immediately into one
     segment, including disconnected/island blocks; multipart output is allowed.

3. zones_5_pop > 1,000:
   - build a strict same-zone rook graph among non-open-space blocks;
   - degree-zero blocks are retained as standalone island_deferred segments;
   - all non-island connected components are processed independently with the
     iterative population/building/compactness algorithm.

4. Zero-population blocks are valid and remain eligible for merging.

5. A connected component whose total population is below 500 is merged as far
   as possible and its final under-500 result is explicitly flagged.

Run the companion topology-diagnostic script first. If a precision-grid test is
judged appropriate, set PRECISION_GRID_M below (for example, 0.05). The source
GeoPackage is never modified.
"""

from __future__ import annotations

import math
import os
import random
import re
import traceback
from dataclasses import dataclass, field

import geopandas as gpd
import numpy as np
import pandas as pd
import pyogrio
import shapely


# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------

IN_GPKG = r"E:\_johannesburg\_analysis\segments_v2\blocks_5_with_building_stats.gpkg"
IN_LAYER = "blocks_5_with_building_stats"  

OUT_FOLDER = (
    r"E:\_johannesburg\_analysis\segments_v2"
    r"\citywide_merge_experiment"
)
OUT_GPKG = os.path.join(OUT_FOLDER, "blocks_5_citywide_merged.gpkg")


# ---------------------------------------------------------------------------
# Fields
# ---------------------------------------------------------------------------

BLOCK_ID = "block_id"
ZONE_ID = "zones_5_ID"
ZONE_POP = "zones_5_pop"
POP = "population"
OPEN_SPACE = "open_space"
AIRPORT = "airport"
BLOCK_AREA = "block_area_m2"
BLDG_COUNT = "bldg_count"
BLDG_AREA_SUM = "bldg_area_sum"


# ---------------------------------------------------------------------------
# Geometry / topology parameters
# ---------------------------------------------------------------------------

# Set to None for raw geometry. After reviewing diagnostics, change to a tested
# value such as 0.05 to round coordinates to a 5-cm precision grid in memory.
PRECISION_GRID_M = 0.01

# If a precision grid is used, write the precision-cleaned geometry to the
# output layers. The source file is still untouched.
OUTPUT_WORKING_GEOMETRY = True

MIN_SHARED_EDGE_M = 0.05


# ---------------------------------------------------------------------------
# Population and candidate-scoring parameters
# ---------------------------------------------------------------------------

MIN_POP = 500.0
SOFT_MAX_POP = 1000.0
SMALL_ZONE_MAX_POP = 1000.0

# <=20% difference receives no building-heterogeneity penalty.
HETERO_TOL = 0.20

# Stabilize relative differences near zero.
AREA_DENSITY_FLOOR = 0.02
COUNT_DENSITY_FLOOR = 2.0

# Tiny-neck threshold.
MIN_SHARED_PERIM_RATIO = 0.03

# Adjustable candidate-score weights.
W_STRAND = 5.0
W_HETERO = 1.5
W_COMPACT = 1.0
W_POP = 0.15
W_NECK = 0.25
W_OVERFLOW = 2.0

# Randomized near-greedy search.
N_RUNS = 20
BASE_SEED = 102
NEAR_BEST_BAND = 0.03

# zones_5_pop should be constant within a zone.
ZONE_POP_CONSISTENCY_TOL = 1e-6

USE_ARROW = False

OPEN_SEGMENT_TYPES = ("open_space", "open_space_airport")
AIRPORT_SEGMENT_TYPES = ("airport", "open_space_airport")
MERGE_EXCLUDED_SEGMENT_TYPES = (
    "open_space",
    "airport",
    "open_space_airport",
)


# ---------------------------------------------------------------------------
# Region state
# ---------------------------------------------------------------------------


@dataclass
class Region:
    members: set[int]
    pop: float
    area: float
    geom_area: float
    perimeter: float
    bldg_count: float
    bldg_area_sum: float
    neighbors: dict[int, float] = field(default_factory=dict)


# ---------------------------------------------------------------------------
# Generic helpers
# ---------------------------------------------------------------------------


def log(message: str) -> None:
    print(message, flush=True)


def resolve_layer(path: str, requested: str) -> str:
    requested = requested.split(".")[-1]
    names = [str(row[0]) for row in pyogrio.list_layers(path)]
    for name in names:
        if name.lower() == requested.lower():
            return name
    raise ValueError(f"Layer '{requested}' not found. Available layers: {names}")


def resolve_field(columns, requested: str) -> str:
    lookup = {str(c).lower(): str(c) for c in columns}
    try:
        return lookup[requested.lower()]
    except KeyError as exc:
        raise ValueError(
            f"Missing field '{requested}'. Available fields:\n{list(columns)}"
        ) from exc


def safe_zone_text(value) -> str:
    if pd.isna(value):
        return "NULL"
    text = str(value)
    if text.endswith(".0"):
        text = text[:-2]
    text = re.sub(r"[^0-9A-Za-z_-]+", "_", text)
    return text or "EMPTY"


def normalize_binary_flag(series: pd.Series, field_name: str) -> pd.Series:
    values = pd.to_numeric(series, errors="coerce").fillna(0)
    bad = ~values.isin([0, 1])
    if bad.any():
        examples = sorted(values.loc[bad].unique().tolist())[:10]
        raise ValueError(
            f"{field_name} must contain only 0, 1, or null. "
            f"Unexpected values include: {examples}"
        )
    return values.eq(1)


def excluded_segment_type(is_open_space: bool, is_airport: bool) -> str:
    if is_open_space and is_airport:
        return "open_space_airport"
    if is_open_space:
        return "open_space"
    if is_airport:
        return "airport"
    raise ValueError("excluded_segment_type called for a non-excluded block.")


def compactness(area: float, perimeter: float) -> float:
    if area <= 0 or perimeter <= 0:
        return 0.0
    return float(np.clip(4.0 * math.pi * area / perimeter**2, 0.0, 1.0))


def rel_diff(a: float, b: float, floor: float) -> float:
    return abs(a - b) / max(abs(a), abs(b), floor)


def area_density(region: Region) -> float:
    return (
        region.bldg_area_sum / region.area
        if region.area > 0
        else 0.0
    )


def count_density(region: Region) -> float:
    return (
        region.bldg_count / (region.area / 10_000.0)
        if region.area > 0
        else 0.0
    )


def population_status(population: float) -> str:
    if population < MIN_POP - 1e-9:
        return "under_500"
    if population > SOFT_MAX_POP + 1e-9:
        return "over_1000"
    return "500_to_1000"


def write_table(df: pd.DataFrame, csv_name: str, gpkg_layer: str) -> None:
    csv_path = os.path.join(OUT_FOLDER, csv_name)
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    if len(df):
        try:
            pyogrio.write_dataframe(
                df,
                OUT_GPKG,
                layer=gpkg_layer,
                driver="GPKG",
                use_arrow=USE_ARROW,
            )
        except Exception as exc:
            log(
                f"Warning: could not write nonspatial GPKG table "
                f"'{gpkg_layer}': {exc}"
            )


# ---------------------------------------------------------------------------
# Rook graph
# ---------------------------------------------------------------------------


def build_rook_graph(gdf: gpd.GeoDataFrame) -> dict[int, dict[int, float]]:
    """Build a strict rook graph for one zone's non-open-space, non-aport blocks."""
    graph = {i: {} for i in range(len(gdf))}
    geoms = gdf.geometry.to_numpy()
    boundaries = np.asarray(shapely.boundary(geoms), dtype=object)
    sindex = gdf.sindex

    for i, geom in enumerate(geoms):
        for j0 in sindex.query(geom, predicate="intersects"):
            j = int(j0)
            if j <= i:
                continue
            shared = float(
                shapely.length(
                    shapely.intersection(boundaries[i], boundaries[j])
                )
            )
            if shared > MIN_SHARED_EDGE_M:
                graph[i][j] = shared
                graph[j][i] = shared

    return graph


def connected_components(graph: dict[int, dict[int, float]]) -> list[list[int]]:
    unseen = set(graph)
    result: list[list[int]] = []

    while unseen:
        start = min(unseen)
        unseen.remove(start)
        stack = [start]
        comp = []

        while stack:
            i = stack.pop()
            comp.append(i)
            for j in graph[i]:
                if j in unseen:
                    unseen.remove(j)
                    stack.append(j)

        result.append(sorted(comp))

    return sorted(result, key=lambda members: min(members))


def extract_component_graph(
    graph: dict[int, dict[int, float]],
    members: list[int],
) -> tuple[list[int], dict[int, dict[int, float]]]:
    """Reindex one component to 0..n-1 for the iterative algorithm."""
    order = sorted(members)
    old_to_new = {old: new for new, old in enumerate(order)}
    member_set = set(order)
    local_graph = {
        old_to_new[old]: {
            old_to_new[nbr]: length
            for nbr, length in graph[old].items()
            if nbr in member_set
        }
        for old in order
    }
    return order, local_graph


# ---------------------------------------------------------------------------
# Iterative merge scoring
# ---------------------------------------------------------------------------


def initial_regions(
    pop: np.ndarray,
    area: np.ndarray,
    geom_area: np.ndarray,
    perimeter: np.ndarray,
    count: np.ndarray,
    area_sum: np.ndarray,
    graph: dict[int, dict[int, float]],
) -> dict[int, Region]:
    return {
        i: Region(
            members={i},
            pop=float(pop[i]),
            area=float(area[i]),
            geom_area=float(geom_area[i]),
            perimeter=float(perimeter[i]),
            bldg_count=float(count[i]),
            bldg_area_sum=float(area_sum[i]),
            neighbors=dict(graph[i]),
        )
        for i in range(len(pop))
    }


def preferred_count(regions: dict[int, Region], rid: int) -> int:
    region = regions[rid]
    return sum(
        region.pop + regions[n].pop <= SOFT_MAX_POP + 1e-9
        for n in region.neighbors
    )


def building_penalty(a: Region, b: Region):
    da = rel_diff(
        area_density(a), area_density(b), AREA_DENSITY_FLOOR
    )
    dc = rel_diff(
        count_density(a), count_density(b), COUNT_DENSITY_FLOOR
    )

    # No penalty inside the practical-equivalence band.
    ea = max(0.0, da - HETERO_TOL) / (1.0 - HETERO_TOL)
    ec = max(0.0, dc - HETERO_TOL) / (1.0 - HETERO_TOL)

    # Downweight the practical effect when one region is very small.
    mix = (
        4.0 * a.area * b.area / (a.area + b.area) ** 2
        if a.area + b.area > 0
        else 0.0
    )
    penalty = mix * 0.5 * (ea**2 + ec**2)

    return penalty, da, dc, (da <= HETERO_TOL and dc <= HETERO_TOL)


def simulated_pref_count(
    regions: dict[int, Region],
    uid: int,
    rid: int,
    sid: int,
    merged_pop: float,
) -> int:
    u = regions[uid]
    nbrs = set(u.neighbors)
    touches_merge = rid in nbrs or sid in nbrs
    nbrs.discard(rid)
    nbrs.discard(sid)

    count = sum(
        u.pop + regions[n].pop <= SOFT_MAX_POP + 1e-9
        for n in nbrs
    )
    if touches_merge and u.pop + merged_pop <= SOFT_MAX_POP + 1e-9:
        count += 1
    return count


def stranding_penalty(
    regions: dict[int, Region],
    rid: int,
    sid: int,
    merged_pop: float,
    merged_nbrs: set[int],
):
    affected = (
        set(regions[rid].neighbors) | set(regions[sid].neighbors)
    ) - {rid, sid}

    penalty = 0.0
    zeroed = 0

    for uid in affected:
        if regions[uid].pop >= MIN_POP:
            continue

        before = preferred_count(regions, uid)
        after = simulated_pref_count(regions, uid, rid, sid, merged_pop)

        if before > 0 and after == 0:
            penalty += 1.0
            zeroed += 1
        elif before > 1 and after == 1:
            penalty += 0.25

    self_options = None
    if merged_pop < MIN_POP:
        self_options = sum(
            merged_pop + regions[n].pop <= SOFT_MAX_POP + 1e-9
            for n in merged_nbrs
        )
        if self_options == 0:
            penalty += 1.5
        elif self_options == 1:
            penalty += 0.25

    return penalty, zeroed, self_options


def population_penalty(pop: float) -> float:
    if pop < MIN_POP:
        return (MIN_POP - pop) / MIN_POP
    if pop <= SOFT_MAX_POP:
        return 0.25 * (pop - MIN_POP) / (SOFT_MAX_POP - MIN_POP)
    return 0.25


def candidate_score(
    regions: dict[int, Region], rid: int, sid: int
) -> dict:
    r, s = regions[rid], regions[sid]
    shared = r.neighbors[sid]
    new_pop = r.pop + s.pop
    new_geom_area = r.geom_area + s.geom_area
    new_perimeter = max(
        1e-9, r.perimeter + s.perimeter - 2.0 * shared
    )
    new_compactness = compactness(new_geom_area, new_perimeter)

    hetero, da, dc, equivalent = building_penalty(r, s)

    new_nbrs = (set(r.neighbors) | set(s.neighbors)) - {rid, sid}
    strand, zeroed, self_options = stranding_penalty(
        regions, rid, sid, new_pop, new_nbrs
    )

    shared_ratio = shared / max(1e-9, min(r.perimeter, s.perimeter))
    neck = max(0.0, MIN_SHARED_PERIM_RATIO - shared_ratio) / (
        MIN_SHARED_PERIM_RATIO
    )
    overflow = max(0.0, new_pop - SOFT_MAX_POP)
    pop_pen = population_penalty(new_pop)

    score = (
        W_STRAND * strand
        + W_HETERO * hetero
        + W_COMPACT * (1.0 - new_compactness)
        + W_POP * pop_pen
        + W_NECK * neck
        + W_OVERFLOW * (overflow / MIN_POP)
    )

    return {
        "candidate": sid,
        "score": score,
        "merged_population": new_pop,
        "merged_compactness": new_compactness,
        "shared_edge_m": shared,
        "shared_perimeter_ratio": shared_ratio,
        "building_penalty": hetero,
        "area_density_rel_diff": da,
        "count_density_rel_diff": dc,
        "building_equivalent_20pct": equivalent,
        "stranding_penalty": strand,
        "newly_stranded_neighbors": zeroed,
        "merged_self_preferred_options": self_options,
        "overflow_excess": overflow,
        "population_penalty": pop_pen,
        "neck_penalty": neck,
    }


def merge_regions(regions: dict[int, Region], rid: int, sid: int) -> None:
    r, s = regions[rid], regions[sid]
    shared = r.neighbors[sid]
    new_nbr_ids = (set(r.neighbors) | set(s.neighbors)) - {rid, sid}

    new_neighbors = {
        n: r.neighbors.get(n, 0.0) + s.neighbors.get(n, 0.0)
        for n in new_nbr_ids
    }

    for n, length in new_neighbors.items():
        regions[n].neighbors.pop(rid, None)
        regions[n].neighbors.pop(sid, None)
        regions[n].neighbors[rid] = length

    r.members |= s.members
    r.pop += s.pop
    r.area += s.area
    r.geom_area += s.geom_area
    r.perimeter = max(
        1e-9, r.perimeter + s.perimeter - 2.0 * shared
    )
    r.bldg_count += s.bldg_count
    r.bldg_area_sum += s.bldg_area_sum
    r.neighbors = new_neighbors

    del regions[sid]


def run_once(
    seed: int,
    arrays: tuple[np.ndarray, ...],
    graph: dict[int, dict[int, float]],
):
    rng = random.Random(seed)
    regions = initial_regions(*arrays, graph)
    blocked: set[int] = set()
    rows = []
    iteration = 0

    while True:
        under = [
            rid
            for rid, region in regions.items()
            if region.pop < MIN_POP - 1e-9 and rid not in blocked
        ]
        if not under:
            break

        # Most constrained first; random value only resolves exact ties.
        target = min(
            (
                preferred_count(regions, rid),
                len(regions[rid].neighbors),
                regions[rid].pop,
                rng.random(),
                rid,
            )
            for rid in under
        )[-1]

        r = regions[target]
        if not r.neighbors:
            blocked.add(target)
            continue

        preferred = [
            n
            for n in r.neighbors
            if r.pop + regions[n].pop <= SOFT_MAX_POP + 1e-9
        ]
        candidates = preferred if preferred else list(r.neighbors)
        forced_overflow = not bool(preferred)

        evaluated = [
            candidate_score(regions, target, n) for n in candidates
        ]
        best_score = min(x["score"] for x in evaluated)
        near_best = [
            x
            for x in evaluated
            if x["score"] <= best_score + NEAR_BEST_BAND
        ]
        chosen = rng.choice(near_best)
        sid = int(chosen["candidate"])

        iteration += 1
        row = {
            "iteration": iteration,
            "seed": seed,
            "target_region": target,
            "candidate_region": sid,
            "target_blocks_before": len(r.members),
            "candidate_blocks_before": len(regions[sid].members),
            "target_pop_before": r.pop,
            "candidate_pop_before": regions[sid].pop,
            "target_area_density": area_density(r),
            "candidate_area_density": area_density(regions[sid]),
            "target_count_density": count_density(r),
            "candidate_count_density": count_density(regions[sid]),
            "target_preferred_options": preferred_count(regions, target),
            "candidate_count_considered": len(evaluated),
            "near_best_candidate_count": len(near_best),
            "forced_overflow": forced_overflow,
        }
        row.update(chosen)
        rows.append(row)

        merge_regions(regions, target, sid)

    return regions, pd.DataFrame(rows)


def summarize_run(seed: int, regions: dict[int, Region], merge_log: pd.DataFrame):
    pops = np.array([region.pop for region in regions.values()])
    compacts = np.array(
        [
            compactness(region.geom_area, region.perimeter)
            for region in regions.values()
        ]
    )
    under = pops < MIN_POP - 1e-9
    over = pops > SOFT_MAX_POP + 1e-9

    return {
        "seed": seed,
        "segment_count": len(regions),
        "under_500_count": int(under.sum()),
        "under_500_deficit": float(
            np.maximum(0.0, MIN_POP - pops).sum()
        ),
        "over_1000_count": int(over.sum()),
        "over_1000_excess": float(
            np.maximum(0.0, pops - SOFT_MAX_POP).sum()
        ),
        "maximum_population": float(pops.max()),
        "compactness_p10": float(np.quantile(compacts, 0.10)),
        "compactness_median": float(np.median(compacts)),
        "mean_building_penalty": (
            float(merge_log["building_penalty"].mean())
            if len(merge_log)
            else 0.0
        ),
    }


def selection_key(summary: dict):
    return (
        summary["under_500_count"],
        summary["under_500_deficit"],
        summary["over_1000_excess"],
        summary["over_1000_count"],
        summary["maximum_population"],
        -summary["compactness_p10"],
        -summary["compactness_median"],
        summary["mean_building_penalty"],
    )


# ---------------------------------------------------------------------------
# Main processing
# ---------------------------------------------------------------------------


def main() -> None:
    os.makedirs(OUT_FOLDER, exist_ok=True)

    layer = resolve_layer(IN_GPKG, IN_LAYER)
    log(f"Reading {IN_GPKG} | layer={layer}")
    gdf = pyogrio.read_dataframe(
        IN_GPKG,
        layer=layer,
        use_arrow=USE_ARROW,
        force_2d=True,
    ).reset_index(drop=True)

    if gdf.crs is None or not gdf.crs.is_projected:
        raise ValueError("Input must use a projected CRS, such as UTM 35S.")

    block_col = resolve_field(gdf.columns, BLOCK_ID)
    zone_col = resolve_field(gdf.columns, ZONE_ID)
    zone_pop_col = resolve_field(gdf.columns, ZONE_POP)
    pop_col = resolve_field(gdf.columns, POP)
    open_col = resolve_field(gdf.columns, OPEN_SPACE)
    airport_col = resolve_field(gdf.columns, AIRPORT)
    area_col = resolve_field(gdf.columns, BLOCK_AREA)
    count_col = resolve_field(gdf.columns, BLDG_COUNT)
    area_sum_col = resolve_field(gdf.columns, BLDG_AREA_SUM)

    gdf["source_row_ix"] = np.arange(len(gdf), dtype=np.int64)

    if gdf[block_col].isna().any() or gdf[block_col].duplicated().any():
        raise ValueError("block_id must be non-null and unique.")
    if gdf[zone_col].isna().any():
        raise ValueError("zones_5_ID contains null values.")

    population = pd.to_numeric(gdf[pop_col], errors="raise").to_numpy(float)
    block_area = pd.to_numeric(gdf[area_col], errors="raise").to_numpy(float)
    bldg_count = (
        pd.to_numeric(gdf[count_col], errors="coerce")
        .fillna(0.0)
        .to_numpy(float)
    )
    bldg_area_sum = (
        pd.to_numeric(gdf[area_sum_col], errors="coerce")
        .fillna(0.0)
        .to_numpy(float)
    )
    zone_pop_all = pd.to_numeric(
        gdf[zone_pop_col], errors="coerce"
    ).to_numpy(float)

    if np.isnan(population).any() or (population < 0).any():
        raise ValueError("population must be non-null and >= 0. Zero is allowed.")
    if np.isnan(block_area).any() or (block_area <= 0).any():
        raise ValueError("block_area_m2 must be non-null and > 0.")
    if (bldg_count < 0).any() or (bldg_area_sum < 0).any():
        raise ValueError("Building counts and areas must be >= 0.")

    open_flags = normalize_binary_flag(
        gdf[open_col], OPEN_SPACE
    ).to_numpy(bool)
    
    airport_flags = normalize_binary_flag(
        gdf[airport_col], AIRPORT
    ).to_numpy(bool)
    
    merge_excluded_flags = open_flags | airport_flags
    eligible_flags = ~merge_excluded_flags
    
    zone_values = gdf[zone_col].to_numpy()
    block_ids = gdf[block_col].astype(str).to_numpy()

    original_geoms = gdf.geometry.to_numpy()
    if gdf.geometry.isna().any() or shapely.is_empty(original_geoms).any():
        raise ValueError("Input contains null or empty geometry.")
    if (~shapely.is_valid(original_geoms)).any():
        raise ValueError(
            "Input contains invalid geometry. Run and review the topology "
            "diagnostic / repair workflow first."
        )

    original_geom_area = shapely.area(original_geoms).astype(float)
    working_geoms = np.asarray(original_geoms, dtype=object)

    if PRECISION_GRID_M is not None:
        if PRECISION_GRID_M <= 0:
            raise ValueError("PRECISION_GRID_M must be positive or None.")
        log(f"Applying in-memory precision grid: {PRECISION_GRID_M} m")
        working_geoms = np.asarray(
            shapely.set_precision(
                original_geoms,
                PRECISION_GRID_M,
                mode="valid_output",
            ),
            dtype=object,
        )
        if shapely.is_empty(working_geoms).any():
            bad = np.flatnonzero(shapely.is_empty(working_geoms))[:20]
            raise ValueError(
                "Precision cleaning collapsed one or more polygons. "
                f"First affected source_row_ix values: {bad.tolist()}"
            )
        if (~shapely.is_valid(working_geoms)).any():
            raise ValueError("Precision-cleaned geometry is not fully valid.")

    working_geom_area = shapely.area(working_geoms).astype(float)

    work = gdf.copy()
    work.geometry = working_geoms

    log(f"Blocks: {len(work):,}")
    log(f"Zones: {work[zone_col].nunique():,}")
    log(f"Open-space blocks: {int(open_flags.sum()):,}")
    log(f"Airport blocks: {int(airport_flags.sum()):,}")
    log(f"Merge-excluded blocks: {int(merge_excluded_flags.sum()):,}")
    log(f"Eligible merge blocks: {int(eligible_flags.sum()):,}")
    log(f"Zero-population blocks: {int((population == 0).sum()):,}")
    log(
        "Zero-population eligible merge blocks: "
        f"{int((eligible_flags & (population == 0)).sum()):,}"
    )

    segment_groups: list[dict] = []
    run_summary_rows: list[dict] = []
    best_log_parts: list[pd.DataFrame] = []
    component_rows: list[dict] = []
    zone_process_rows: list[dict] = []

    zones = sorted(pd.unique(zone_values), key=lambda x: str(x))

    for zone_number, zone in enumerate(zones, start=1):
        zone_idx = np.flatnonzero(zone_values == zone)

        zone_open_idx = zone_idx[open_flags[zone_idx]]
        zone_airport_idx = zone_idx[airport_flags[zone_idx]]
        zone_excluded_idx = zone_idx[merge_excluded_flags[zone_idx]]
        zone_eligible_idx = zone_idx[eligible_flags[zone_idx]]

        zone_pop_values = pd.Series(zone_pop_all[zone_idx]).dropna().to_numpy()
        if len(zone_pop_values) == 0:
            raise ValueError(f"Zone {zone} has no non-null zones_5_pop value.")
        if float(np.max(zone_pop_values) - np.min(zone_pop_values)) > (
            ZONE_POP_CONSISTENCY_TOL
        ):
            raise ValueError(
                f"Zone {zone} has inconsistent zones_5_pop values: "
                f"min={np.min(zone_pop_values)}, max={np.max(zone_pop_values)}"
            )
        zone_pop_value = float(np.median(zone_pop_values))

        log(
            f"Zone {zone_number:03d}/{len(zones):03d} | "
            f"ID={zone} | blocks={len(zone_idx):,} | "
            f"eligible_merge={len(zone_eligible_idx):,} | "
            f"open_space={len(zone_open_idx):,} | "
            f"airport={len(zone_airport_idx):,} | "
            f"merge_excluded={len(zone_excluded_idx):,} | "
            f"zones_5_pop={zone_pop_value:,.3f}"
        )

        # Open-space and airport blocks always remain standalone.
        for global_ix in zone_excluded_idx:
            stype = excluded_segment_type(
                bool(open_flags[global_ix]),
                bool(airport_flags[global_ix]),
            )
        
            segment_groups.append(
                {
                    "members": [int(global_ix)],
                    "zone_id": zone,
                    "zones_5_pop": zone_pop_value,
                    "component_id": stype,
                    "segment_type": stype,
                    "selected_seed": None,
                    "component_total_population": float(population[global_ix]),
                }
            )

        if len(zone_eligible_idx) == 0:
            zone_process_rows.append(
                {
                    "zone_id": zone,
                    "zones_5_pop": zone_pop_value,
                    "processing_branch": "merge_excluded_only",
                    "total_blocks": len(zone_idx),
                    "eligible_merge_blocks": 0,
                    "eligible_non_open_blocks": 0,
                    "open_space_blocks": len(zone_open_idx),
                    "airport_blocks": len(zone_airport_idx),
                    "merge_excluded_blocks": len(zone_excluded_idx),
                    "rook_components": 0,
                    "islands_deferred": 0,
                    "iterative_components": 0,
                }
            )
            continue

        # ---------------------------------------------------------------
        # Small-zone shortcut: one multipart segment for all eligible merge blocks.
        # ---------------------------------------------------------------
        if zone_pop_value <= SMALL_ZONE_MAX_POP + 1e-9:
            segment_groups.append(
                {
                    "members": zone_eligible_idx.astype(int).tolist(),
                    "zone_id": zone,
                    "zones_5_pop": zone_pop_value,
                    "component_id": "all_eligible_merge",
                    "segment_type": "auto_small_zone",
                    "selected_seed": None,
                    "component_total_population": float(
                        population[zone_eligible_idx].sum()
                    ),
                }
            )
            component_rows.append(
                {
                    "zone_id": zone,
                    "component_id": "all_eligible_merge",
                    "processing_branch": "auto_small_zone",
                    "block_count": len(zone_eligible_idx),
                    "population": float(population[zone_eligible_idx].sum()),
                    "rook_edge_count": None,
                    "is_island": False,
                    "below_500_unavoidable": float(
                        population[zone_eligible_idx].sum()
                    ) < MIN_POP,
                    "selected_seed": None,
                    "final_segment_count": 1,
                }
            )
            zone_process_rows.append(
                {
                    "zone_id": zone,
                    "zones_5_pop": zone_pop_value,
                    "processing_branch": "auto_small_zone",
                    "total_blocks": len(zone_idx),
                    "eligible_merge_blocks": len(zone_eligible_idx),
                    "eligible_non_open_blocks": len(zone_eligible_idx),
                    "open_space_blocks": len(zone_open_idx),
                    "airport_blocks": len(zone_airport_idx),
                    "merge_excluded_blocks": len(zone_excluded_idx),
                    "rook_components": None,
                    "islands_deferred": 0,
                    "iterative_components": 0,
                }
            )
            continue

        # ---------------------------------------------------------------
        # Large zone: strict rook graph among eligible merge blocks.
        # ---------------------------------------------------------------
        zone_eligible = work.iloc[zone_eligible_idx].copy().reset_index(drop=True)
        zone_eligible["global_ix"] = zone_eligible_idx
        graph = build_rook_graph(zone_eligible)
        comps = connected_components(graph)

        island_count = 0
        iterative_component_count = 0

        for comp_no, members in enumerate(comps, start=1):
            component_id = f"z{safe_zone_text(zone)}_c{comp_no:03d}"
            local_global = zone_eligible.loc[members, "global_ix"].to_numpy(int)
            comp_pop = float(population[local_global].sum())
            edge_count = sum(len(graph[i]) for i in members) // 2
            is_island = len(members) == 1 and len(graph[members[0]]) == 0

            if is_island:
                island_count += 1
                global_ix = int(local_global[0])
                segment_groups.append(
                    {
                        "members": [global_ix],
                        "zone_id": zone,
                        "zones_5_pop": zone_pop_value,
                        "component_id": component_id,
                        "segment_type": "island_deferred",
                        "selected_seed": None,
                        "component_total_population": comp_pop,
                    }
                )
                component_rows.append(
                    {
                        "zone_id": zone,
                        "component_id": component_id,
                        "processing_branch": "island_deferred",
                        "block_count": 1,
                        "population": comp_pop,
                        "rook_edge_count": 0,
                        "is_island": True,
                        "below_500_unavoidable": comp_pop < MIN_POP,
                        "selected_seed": None,
                        "final_segment_count": 1,
                    }
                )
                continue

            iterative_component_count += 1
            order, comp_graph = extract_component_graph(graph, members)
            comp_global = zone_eligible.loc[order, "global_ix"].to_numpy(int)

            arrays = (
                population[comp_global],
                block_area[comp_global],
                working_geom_area[comp_global],
                shapely.length(working_geoms[comp_global]).astype(float),
                bldg_count[comp_global],
                bldg_area_sum[comp_global],
            )

            best = None
            component_run_rows = []

            for run_no in range(N_RUNS):
                seed = BASE_SEED + run_no
                regions, merge_log = run_once(seed, arrays, comp_graph)
                summary = summarize_run(seed, regions, merge_log)
                summary.update(
                    {
                        "run_number": run_no + 1,
                        "zone_id": zone,
                        "component_id": component_id,
                        "component_block_count": len(comp_global),
                        "component_population": comp_pop,
                    }
                )
                component_run_rows.append(summary)

                key = selection_key(summary)
                if best is None or key < best[0]:
                    best = (key, seed, regions, merge_log)

            assert best is not None
            _, best_seed, best_regions, best_log = best

            for summary in component_run_rows:
                summary["selected"] = summary["seed"] == best_seed
                run_summary_rows.append(summary)

            if len(best_log):
                best_log = best_log.copy()
                best_log.insert(0, "zone_id", zone)
                best_log.insert(1, "component_id", component_id)
                best_log["target_anchor_source_row_ix"] = best_log[
                    "target_region"
                ].map(lambda x: int(comp_global[int(x)]))
                best_log["candidate_anchor_source_row_ix"] = best_log[
                    "candidate_region"
                ].map(lambda x: int(comp_global[int(x)]))
                best_log["target_anchor_block_id"] = best_log[
                    "target_anchor_source_row_ix"
                ].map(lambda x: block_ids[int(x)])
                best_log["candidate_anchor_block_id"] = best_log[
                    "candidate_anchor_source_row_ix"
                ].map(lambda x: block_ids[int(x)])
                best_log_parts.append(best_log)

            for rid in sorted(
                best_regions,
                key=lambda x: min(best_regions[x].members),
            ):
                region = best_regions[rid]
                member_global = [
                    int(comp_global[local_ix])
                    for local_ix in sorted(region.members)
                ]
                if comp_pop < MIN_POP:
                    segment_type = "unavoidable_under500_component"
                elif region.pop < MIN_POP - 1e-9:
                    segment_type = "iterative_under500_unresolved"
                else:
                    segment_type = "iterative"

                segment_groups.append(
                    {
                        "members": member_global,
                        "zone_id": zone,
                        "zones_5_pop": zone_pop_value,
                        "component_id": component_id,
                        "segment_type": segment_type,
                        "selected_seed": best_seed,
                        "component_total_population": comp_pop,
                    }
                )

            selected_summary = next(
                row
                for row in component_run_rows
                if row["seed"] == best_seed
            )
            component_rows.append(
                {
                    "zone_id": zone,
                    "component_id": component_id,
                    "processing_branch": "iterative",
                    "block_count": len(comp_global),
                    "population": comp_pop,
                    "rook_edge_count": edge_count,
                    "is_island": False,
                    "below_500_unavoidable": comp_pop < MIN_POP,
                    "selected_seed": best_seed,
                    "final_segment_count": selected_summary[
                        "segment_count"
                    ],
                    "final_under_500_count": selected_summary[
                        "under_500_count"
                    ],
                    "final_over_1000_count": selected_summary[
                        "over_1000_count"
                    ],
                    "final_over_1000_excess": selected_summary[
                        "over_1000_excess"
                    ],
                }
            )

        zone_process_rows.append(
            {
                "zone_id": zone,
                "zones_5_pop": zone_pop_value,
                "processing_branch": "iterative_large_zone",
                "total_blocks": len(zone_idx),
                "eligible_merge_blocks": len(zone_eligible_idx),
                "eligible_non_open_blocks": len(zone_eligible_idx),
                "open_space_blocks": len(zone_open_idx),
                "airport_blocks": len(zone_airport_idx),
                "merge_excluded_blocks": len(zone_excluded_idx),
                "rook_components": len(comps),
                "islands_deferred": island_count,
                "iterative_components": iterative_component_count,
            }
        )

    # ------------------------------------------------------------------
    # Assign stable citywide segment IDs.
    # ------------------------------------------------------------------

    segment_groups.sort(
        key=lambda x: (
            str(x["zone_id"]),
            min(x["members"]),
            x["segment_type"],
        )
    )

    zone_counter: dict[str, int] = {}
    block_to_segment: dict[int, str] = {}
    segment_meta_rows = []

    for group in segment_groups:
        zone_text = safe_zone_text(group["zone_id"])
        zone_counter[zone_text] = zone_counter.get(zone_text, 0) + 1
        segment_id = f"z{zone_text}_s{zone_counter[zone_text]:04d}"

        for global_ix in group["members"]:
            if global_ix in block_to_segment:
                raise RuntimeError(
                    f"Block source_row_ix={global_ix} assigned more than once."
                )
            block_to_segment[global_ix] = segment_id

        stype = group["segment_type"]
        
        segment_meta_rows.append(
            {
                "segment_id": segment_id,
                "zone_id": group["zone_id"],
                "zones_5_pop": group["zones_5_pop"],
                "component_id": group["component_id"],
                "segment_type": stype,
                "selected_seed": group["selected_seed"],
                "component_total_population": group[
                    "component_total_population"
                ],
                "minimum_source_row_ix": min(group["members"]),
                "merge_excluded_segment": int(
                    stype in MERGE_EXCLUDED_SEGMENT_TYPES
                ),
                "open_space_segment": int(stype in OPEN_SEGMENT_TYPES),
                "airport_segment": int(stype in AIRPORT_SEGMENT_TYPES),
            }
        )

    if len(block_to_segment) != len(work):
        missing = sorted(set(range(len(work))) - set(block_to_segment))[:20]
        raise RuntimeError(
            f"Not every block was assigned. First missing indices: {missing}"
        )

    meta = pd.DataFrame(segment_meta_rows)
    meta_lookup = meta.set_index("segment_id")

    # ------------------------------------------------------------------
    # Block output and final dissolve.
    # ------------------------------------------------------------------

    blocks_out = gdf.copy()
    if PRECISION_GRID_M is not None and OUTPUT_WORKING_GEOMETRY:
        blocks_out.geometry = working_geoms

    blocks_out["segment_id"] = [
        block_to_segment[i] for i in range(len(blocks_out))
    ]
    blocks_out["segment_type"] = blocks_out["segment_id"].map(
        meta_lookup["segment_type"]
    )
    blocks_out["component_id"] = blocks_out["segment_id"].map(
        meta_lookup["component_id"]
    )
    blocks_out["selected_seed"] = blocks_out["segment_id"].map(
        meta_lookup["selected_seed"]
    )
    blocks_out["island_deferred"] = blocks_out["segment_type"].eq(
        "island_deferred"
    ).astype(np.int8)
    blocks_out["auto_small_zone"] = blocks_out["segment_type"].eq(
        "auto_small_zone"
    ).astype(np.int8)
    blocks_out["merge_excluded"] = blocks_out["segment_type"].isin(
        MERGE_EXCLUDED_SEGMENT_TYPES
    ).astype(np.int8)
    
    blocks_out["open_space_excluded"] = blocks_out["segment_type"].isin(
        OPEN_SEGMENT_TYPES
    ).astype(np.int8)
    
    blocks_out["airport_excluded"] = blocks_out["segment_type"].isin(
        AIRPORT_SEGMENT_TYPES
    ).astype(np.int8)
    blocks_out["precision_grid_m"] = PRECISION_GRID_M
    blocks_out["geom_area_original_m2"] = original_geom_area
    blocks_out["geom_area_working_m2"] = working_geom_area
    blocks_out["geom_area_delta_m2"] = (
        working_geom_area - original_geom_area
    )

    log("Dissolving final segments...")
    segment_geoms = blocks_out[["segment_id", "geometry"]].dissolve(
        by="segment_id", as_index=False
    )

    aggregations = pd.DataFrame(
        {
            "segment_id": blocks_out["segment_id"],
            "population": population,
            "block_area_m2": block_area,
            "bldg_count": bldg_count,
            "bldg_area_sum": bldg_area_sum,
        }
    ).groupby("segment_id", as_index=False).agg(
        block_count=("population", "size"),
        population=("population", "sum"),
        block_area_m2=("block_area_m2", "sum"),
        bldg_count=("bldg_count", "sum"),
        bldg_area_sum=("bldg_area_sum", "sum"),
    )

    segments = (
        segment_geoms.merge(meta, on="segment_id", validate="one_to_one")
        .merge(aggregations, on="segment_id", validate="one_to_one")
    )

    segments["bldg_count"] = segments["bldg_count"].round().astype("int64")
    segments["bldg_area_density"] = (
        segments["bldg_area_sum"] / segments["block_area_m2"]
    )
    segments["bldg_count_density"] = segments["bldg_count"] / (
        segments["block_area_m2"] / 10_000.0
    )
    segments["pop_status"] = segments["population"].map(population_status)
    segments["pop_shortfall_500"] = np.maximum(
        0.0, MIN_POP - segments["population"]
    )
    segments["pop_excess_1000"] = np.maximum(
        0.0, segments["population"] - SOFT_MAX_POP
    )
    segments["segment_area_geom_m2"] = segments.geometry.area
    segments["segment_perimeter_m"] = segments.geometry.length
    segments["compactness"] = [
        compactness(area, perimeter)
        for area, perimeter in zip(
            segments["segment_area_geom_m2"],
            segments["segment_perimeter_m"],
        )
    ]
    segments["geometry_part_count"] = shapely.get_num_geometries(
        segments.geometry.to_numpy()
    )
    segments["is_multipart"] = (
        segments["geometry_part_count"] > 1
    ).astype(np.int8)
    segments["precision_grid_m"] = PRECISION_GRID_M

    # ------------------------------------------------------------------
    # QA and summaries.
    # ------------------------------------------------------------------

    if not math.isclose(
        float(segments["population"].sum()),
        float(population.sum()),
        abs_tol=1e-6,
        rel_tol=0.0,
    ):
        raise RuntimeError("Citywide population was not preserved.")

    segment_zone_pop = segments.groupby("zone_id")["population"].sum()
    source_zone_pop = pd.Series(population).groupby(zone_values).sum()
    for zone in source_zone_pop.index:
        if not math.isclose(
            float(segment_zone_pop.loc[zone]),
            float(source_zone_pop.loc[zone]),
            abs_tol=1e-6,
            rel_tol=0.0,
        ):
            raise RuntimeError(f"Population was not preserved in zone {zone}.")

    zone_process = pd.DataFrame(zone_process_rows)
    final_zone = segments.groupby("zone_id", as_index=False).agg(
        final_segment_count=("segment_id", "size"),
        final_population=("population", "sum"),
        under_500_segments=(
            "pop_status",
            lambda x: int((x == "under_500").sum()),
        ),
        over_1000_segments=(
            "pop_status",
            lambda x: int((x == "over_1000").sum()),
        ),
        total_overflow=("pop_excess_1000", "sum"),
        island_segments=(
            "segment_type",
            lambda x: int((x == "island_deferred").sum()),
        ),
        open_space_segments=(
            "segment_type",
            lambda x: int(x.isin(OPEN_SEGMENT_TYPES).sum()),
        ),
        airport_segments=(
            "segment_type",
            lambda x: int(x.isin(AIRPORT_SEGMENT_TYPES).sum()),
        ),
        merge_excluded_segments=(
            "segment_type",
            lambda x: int(x.isin(MERGE_EXCLUDED_SEGMENT_TYPES).sum()),
        ),
        multipart_segments=("is_multipart", "sum"),
        median_compactness=("compactness", "median"),
    )
    zone_summary = zone_process.merge(
        final_zone, on="zone_id", how="left", validate="one_to_one"
    )
    zone_summary["sum_source_block_population"] = zone_summary[
        "zone_id"
    ].map(source_zone_pop)
    zone_summary["population_preserved"] = np.isclose(
        zone_summary["final_population"],
        zone_summary["sum_source_block_population"],
        atol=1e-6,
        rtol=0.0,
    )

    component_summary = pd.DataFrame(component_rows)
    run_summary = pd.DataFrame(run_summary_rows)
    merge_log_best = (
        pd.concat(best_log_parts, ignore_index=True)
        if best_log_parts
        else pd.DataFrame()
    )

    geometry_summary = pd.DataFrame(
        [
            {
                "precision_grid_m": PRECISION_GRID_M,
                "output_working_geometry": OUTPUT_WORKING_GEOMETRY,
                "total_original_geometry_area_m2": float(
                    original_geom_area.sum()
                ),
                "total_working_geometry_area_m2": float(
                    working_geom_area.sum()
                ),
                "total_geometry_area_delta_m2": float(
                    working_geom_area.sum() - original_geom_area.sum()
                ),
                "sum_absolute_block_area_change_m2": float(
                    np.abs(working_geom_area - original_geom_area).sum()
                ),
                "maximum_absolute_block_area_change_m2": float(
                    np.abs(working_geom_area - original_geom_area).max()
                ),
            }
        ]
    )

    city_summary = pd.DataFrame(
        [
            {
                "source_blocks": len(blocks_out),
                "source_zones": len(zones),
                "source_population": float(population.sum()),
                "open_space_blocks": int(open_flags.sum()),
                "airport_blocks": int(airport_flags.sum()),
                "merge_excluded_blocks": int(merge_excluded_flags.sum()),
                "eligible_merge_blocks": int(eligible_flags.sum()),
                "zero_population_blocks": int((population == 0).sum()),
                "zero_population_eligible_merge_blocks": int(
                    (eligible_flags & (population == 0)).sum()
                ),
                "final_segments": len(segments),
                "iterative_segments": int(
                    segments["segment_type"].isin(
                        [
                            "iterative",
                            "iterative_under500_unresolved",
                            "unavoidable_under500_component",
                        ]
                    ).sum()
                ),
                "auto_small_zone_segments": int(
                    (segments["segment_type"] == "auto_small_zone").sum()
                ),
                "open_space_segments": int(
                    segments["segment_type"].isin(OPEN_SEGMENT_TYPES).sum()
                ),
                "airport_segments": int(
                    segments["segment_type"].isin(AIRPORT_SEGMENT_TYPES).sum()
                ),
                "merge_excluded_segments": int(
                    segments["segment_type"].isin(MERGE_EXCLUDED_SEGMENT_TYPES).sum()
                ),
                "island_deferred_segments": int(
                    (segments["segment_type"] == "island_deferred").sum()
                ),
                "under_500_segments": int(
                    (segments["pop_status"] == "under_500").sum()
                ),
                "over_1000_segments": int(
                    (segments["pop_status"] == "over_1000").sum()
                ),
                "total_overflow": float(
                    segments["pop_excess_1000"].sum()
                ),
                "median_compactness": float(
                    segments["compactness"].median()
                ),
                "precision_grid_m": PRECISION_GRID_M,
                "runs_per_iterative_component": N_RUNS,
            }
        ]
    )

    # ------------------------------------------------------------------
    # Write outputs.
    # ------------------------------------------------------------------

    if os.path.exists(OUT_GPKG):
        os.remove(OUT_GPKG)

    log(f"Writing output GeoPackage: {OUT_GPKG}")
    pyogrio.write_dataframe(
        segments,
        OUT_GPKG,
        layer="segments_best",
        driver="GPKG",
        use_arrow=USE_ARROW,
        promote_to_multi=True,
    )
    pyogrio.write_dataframe(
        blocks_out,
        OUT_GPKG,
        layer="blocks_with_segment_id",
        driver="GPKG",
        use_arrow=USE_ARROW,
        promote_to_multi=True,
    )

    write_table(city_summary, "city_summary.csv", "city_summary")
    write_table(zone_summary, "zone_summary.csv", "zone_summary")
    write_table(
        component_summary,
        "component_summary.csv",
        "component_summary",
    )
    write_table(run_summary, "run_summary.csv", "run_summary")
    write_table(
        merge_log_best,
        "merge_log_best.csv",
        "merge_log_best",
    )
    write_table(
        segments.drop(columns="geometry"),
        "segment_summary.csv",
        "segment_summary",
    )
    write_table(
        geometry_summary,
        "geometry_processing_summary.csv",
        "geometry_processing_summary",
    )

    log("")
    log("Citywide merge complete.")
    log(f"Output: {OUT_GPKG}")
    log(f"Final segments: {len(segments):,}")
    log(
        "Under 500 (all types): "
        f"{int((segments['pop_status'] == 'under_500').sum()):,}"
    )
    log(
        "Over 1000: "
        f"{int((segments['pop_status'] == 'over_1000').sum()):,}"
    )
    log(
        "Deferred islands: "
        f"{int((segments['segment_type'] == 'island_deferred').sum()):,}"
    )
    log(
        "Standalone open-space segments: "
        f"{int(segments['segment_type'].isin(OPEN_SEGMENT_TYPES).sum()):,}"
    )
    log(
        "Standalone airport segments: "
        f"{int(segments['segment_type'].isin(AIRPORT_SEGMENT_TYPES).sum()):,}"
    )
    log(
        "Total merge-excluded standalone segments: "
        f"{int(segments['segment_type'].isin(MERGE_EXCLUDED_SEGMENT_TYPES).sum()):,}"
    )


if __name__ == "__main__":
    try:
        main()
    except Exception:
        log("ERROR:")
        log(traceback.format_exc())
        raise


In [ ]:
"""
Second-stage resolution of disconnected under-500 island segments.

This script starts from the output of the first-stage citywide rook-merging
workflow. It does NOT reopen or split any first-stage segment. Instead, within
each zones_5_ID / zone_id whose zones_5_pop is greater than 1,000, it attaches
under-500 island-derived segments to nearby same-zone, non-open-space segments.

Core rules
----------
1. Small zones (zones_5_pop <= 1,000) are left unchanged. Their non-open-space
   blocks were already dissolved to one potentially multipart segment in stage 1.
2. Open-space segments are left unchanged and are never targets or candidates.
3. Under-500 segments with a first-stage type of:
       island_deferred
       unavoidable_under500_component
       iterative_under500_unresolved
   initiate stage-2 attachment merges.
4. Island-derived segments already at or above 500 do not initiate a merge, but
   they remain eligible to receive another under-500 island.
5. Candidates are the K nearest same-zone, non-open-space current regions,
   measured by polygon-to-polygon distance. If any of those candidates keeps the
   merged population <= 1,000, over-1,000 candidates are excluded. Otherwise,
   overflow is allowed and penalized.
6. Candidate score (lower is better):
       W_STRAND   * local future-feasibility penalty
     + W_DISTANCE * normalized polygon-distance penalty
     + W_HETERO   * building-heterogeneity penalty
     + W_POP      * population penalty
     + W_OVERFLOW * overflow amount / 500
7. Twenty seeded near-greedy runs are performed independently for each zone
   containing stage-2 targets. The best run prioritizes:
       unresolved under-500 island count,
       unresolved population deficit,
       overflow amount/count,
       maximum and total attachment distance,
       building heterogeneity.

Outputs
-------
A new GeoPackage is written with:
    segments_islands_resolved
    blocks_with_final_segment_id
    island_attachment_lines
and CSV / optional GeoPackage tables for:
    city_island_summary
    zone_island_summary
    island_run_summary
    island_merge_log_best
    island_candidate_audit_best
    stage1_to_final_crosswalk
    final_segment_summary

The source stage-1 GeoPackage is never modified.
"""

from __future__ import annotations

import math
import os
import random
import re
import traceback
import zlib
from dataclasses import dataclass, field

import geopandas as gpd
import numpy as np
import pandas as pd
import pyogrio
import shapely


# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------

IN_GPKG = (
    r"E:\_johannesburg\_analysis\segments_v2"
    r"\citywide_merge_experiment\blocks_5_citywide_merged.gpkg"
)
SEGMENTS_LAYER = "segments_best"
BLOCKS_LAYER = "blocks_with_segment_id"

OUT_FOLDER = (
    r"E:\_johannesburg\_analysis\segments_v2"
    r"\citywide_island_resolution"
)
OUT_GPKG = os.path.join(
    OUT_FOLDER,
    "blocks_5_citywide_islands_resolved.gpkg",
)


# ---------------------------------------------------------------------------
# Expected fields in stage-1 output
# ---------------------------------------------------------------------------

SEGMENT_ID = "segment_id"
ZONE_ID = "zone_id"
ZONE_POP = "zones_5_pop"
SEGMENT_TYPE = "segment_type"
COMPONENT_ID = "component_id"
POP = "population"
BLOCK_COUNT = "block_count"
BLOCK_AREA = "block_area_m2"
BLDG_COUNT = "bldg_count"
BLDG_AREA_SUM = "bldg_area_sum"


# ---------------------------------------------------------------------------
# Eligibility and population parameters
# ---------------------------------------------------------------------------

MIN_POP = 500.0
SOFT_MAX_POP = 1000.0
SMALL_ZONE_MAX_POP = 1000.0

# These first-stage segment types are considered unresolved island origins when
# their population is below 500.
ISLAND_TARGET_TYPES = {
    "island_deferred",
    "unavoidable_under500_component",
    "iterative_under500_unresolved",
}

# Open-space and airport-derived stage-1 segments are never targets or candidates.
OPEN_SEGMENT_TYPES = {"open_space", "open_space_airport"}
AIRPORT_SEGMENT_TYPES = {"airport", "open_space_airport"}
MERGE_EXCLUDED_SEGMENT_TYPES = OPEN_SEGMENT_TYPES | AIRPORT_SEGMENT_TYPES

EXCLUDED_CANDIDATE_TYPES = MERGE_EXCLUDED_SEGMENT_TYPES


# ---------------------------------------------------------------------------
# Candidate neighborhood and scoring parameters
# ---------------------------------------------------------------------------

# Number of nearest same-zone current regions considered as plausible recipients.
K_NEAREST = 8

# <=20% difference on both building measures receives no heterogeneity penalty.
HETERO_TOL = 0.20
AREA_DENSITY_FLOOR = 0.02
COUNT_DENSITY_FLOOR = 2.0

# Adjustable score weights.
W_STRAND = 5.0
W_DISTANCE = 2.0
W_HETERO = 1.5
W_POP = 0.15
W_OVERFLOW = 2.0

# Randomized near-greedy search.
N_RUNS = 20
BASE_SEED = 2026
NEAR_BEST_BAND = 0.03

# Numeric tolerances.
ZONE_POP_CONSISTENCY_TOL = 1e-6
DISTANCE_TIE_TOL = 1e-9

USE_ARROW = False


# ---------------------------------------------------------------------------
# Region state
# ---------------------------------------------------------------------------


@dataclass
class Region:
    """A current stage-2 region made of one or more stage-1 segments."""

    members: set[int]
    pop: float
    area: float
    bldg_count: float
    bldg_area_sum: float
    contains_island_target: bool
    initial_target_count: int
    stage2_merge_count: int = 0


@dataclass
class RunState:
    """Mutable state for one randomized run within one zone."""

    regions: dict[int, Region]
    active: np.ndarray
    distance: np.ndarray
    anchor_left: np.ndarray
    anchor_right: np.ndarray
    blocked: set[int] = field(default_factory=set)


# ---------------------------------------------------------------------------
# Generic helpers
# ---------------------------------------------------------------------------


def log(message: str) -> None:
    print(message, flush=True)


def resolve_layer(path: str, requested: str) -> str:
    requested = requested.split(".")[-1]
    names = [str(row[0]) for row in pyogrio.list_layers(path)]
    for name in names:
        if name.lower() == requested.lower():
            return name
    raise ValueError(f"Layer '{requested}' not found. Available layers: {names}")


def resolve_field(columns, requested: str) -> str:
    lookup = {str(c).lower(): str(c) for c in columns}
    try:
        return lookup[requested.lower()]
    except KeyError as exc:
        raise ValueError(
            f"Missing field '{requested}'. Available fields:\n{list(columns)}"
        ) from exc


def safe_zone_text(value) -> str:
    if pd.isna(value):
        return "NULL"
    text = str(value)
    if text.endswith(".0"):
        text = text[:-2]
    text = re.sub(r"[^0-9A-Za-z_-]+", "_", text)
    return text or "EMPTY"


def stable_zone_seed(zone_value, run_number: int) -> int:
    zone_crc = zlib.crc32(str(zone_value).encode("utf-8")) % 1_000_000
    return int(BASE_SEED + zone_crc * 100 + run_number)


def population_status(population: float) -> str:
    if population < MIN_POP - 1e-9:
        return "under_500"
    if population > SOFT_MAX_POP + 1e-9:
        return "over_1000"
    return "500_to_1000"


def rel_diff(a: float, b: float, floor: float) -> float:
    return abs(a - b) / max(abs(a), abs(b), floor)


def area_density(region: Region) -> float:
    return region.bldg_area_sum / region.area if region.area > 0 else 0.0


def count_density(region: Region) -> float:
    return (
        region.bldg_count / (region.area / 10_000.0)
        if region.area > 0
        else 0.0
    )


def building_penalty(a: Region, b: Region):
    da = rel_diff(area_density(a), area_density(b), AREA_DENSITY_FLOOR)
    dc = rel_diff(count_density(a), count_density(b), COUNT_DENSITY_FLOOR)

    # Only the amount beyond the 20% practical-equivalence band is penalized.
    ea = max(0.0, da - HETERO_TOL) / (1.0 - HETERO_TOL)
    ec = max(0.0, dc - HETERO_TOL) / (1.0 - HETERO_TOL)

    # Reduce practical influence when one region is very small relative to the other.
    mix = (
        4.0 * a.area * b.area / (a.area + b.area) ** 2
        if a.area + b.area > 0
        else 0.0
    )
    penalty = mix * 0.5 * (ea**2 + ec**2)

    return penalty, da, dc, (da <= HETERO_TOL and dc <= HETERO_TOL), mix


def population_penalty(pop: float) -> float:
    if pop < MIN_POP:
        return (MIN_POP - pop) / MIN_POP
    if pop <= SOFT_MAX_POP:
        return 0.25 * (pop - MIN_POP) / (SOFT_MAX_POP - MIN_POP)
    return 0.25


def write_table(
    df: pd.DataFrame,
    csv_name: str,
    gpkg_layer: str,
) -> None:
    csv_path = os.path.join(OUT_FOLDER, csv_name)
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    if len(df):
        try:
            pyogrio.write_dataframe(
                df,
                OUT_GPKG,
                layer=gpkg_layer,
                driver="GPKG",
                use_arrow=USE_ARROW,
            )
        except Exception as exc:
            log(
                f"Warning: could not write nonspatial GPKG table "
                f"'{gpkg_layer}': {exc}"
            )


# ---------------------------------------------------------------------------
# Distance matrix and dynamic proximity graph
# ---------------------------------------------------------------------------


def pairwise_distance_matrix(geoms: np.ndarray) -> np.ndarray:
    """Return a dense symmetric polygon-to-polygon distance matrix."""
    n = len(geoms)
    result = np.full((n, n), np.inf, dtype=float)
    np.fill_diagonal(result, 0.0)

    for i in range(n - 1):
        d = np.asarray(shapely.distance(geoms[i], geoms[i + 1 :]), dtype=float)
        result[i, i + 1 :] = d
        result[i + 1 :, i] = d

    return result


def initial_anchor_matrices(n: int) -> tuple[np.ndarray, np.ndarray]:
    """Store the original stage-1 segment pair that realizes each distance."""
    left = np.repeat(np.arange(n, dtype=np.int32)[:, None], n, axis=1)
    right = np.repeat(np.arange(n, dtype=np.int32)[None, :], n, axis=0)
    return left, right


def initial_regions(
    pop: np.ndarray,
    area: np.ndarray,
    bldg_count: np.ndarray,
    bldg_area_sum: np.ndarray,
    target_origin: np.ndarray,
) -> dict[int, Region]:
    return {
        i: Region(
            members={i},
            pop=float(pop[i]),
            area=float(area[i]),
            bldg_count=float(bldg_count[i]),
            bldg_area_sum=float(bldg_area_sum[i]),
            contains_island_target=bool(target_origin[i]),
            initial_target_count=int(bool(target_origin[i])),
        )
        for i in range(len(pop))
    }


def make_run_state(
    pop: np.ndarray,
    area: np.ndarray,
    bldg_count: np.ndarray,
    bldg_area_sum: np.ndarray,
    target_origin: np.ndarray,
    base_distance: np.ndarray,
    base_anchor_left: np.ndarray,
    base_anchor_right: np.ndarray,
) -> RunState:
    n = len(pop)
    return RunState(
        regions=initial_regions(
            pop,
            area,
            bldg_count,
            bldg_area_sum,
            target_origin,
        ),
        active=np.ones(n, dtype=bool),
        distance=base_distance.copy(),
        anchor_left=base_anchor_left.copy(),
        anchor_right=base_anchor_right.copy(),
    )


def active_ids(state: RunState, exclude: set[int] | None = None) -> np.ndarray:
    ids = np.flatnonzero(state.active)
    if exclude:
        mask = ~np.isin(ids, np.fromiter(exclude, dtype=int))
        ids = ids[mask]
    return ids


def nearest_candidates(
    state: RunState,
    rid: int,
    k: int = K_NEAREST,
) -> list[int]:
    ids = active_ids(state, {rid})
    if len(ids) == 0:
        return []

    d = state.distance[rid, ids]
    # Stable order: distance first, then region ID.
    order = np.lexsort((ids, d))
    return ids[order[: min(k, len(ids))]].astype(int).tolist()


def preferred_candidate_ids(state: RunState, rid: int) -> list[int]:
    r = state.regions[rid]
    return [
        sid
        for sid in nearest_candidates(state, rid)
        if r.pop + state.regions[sid].pop <= SOFT_MAX_POP + 1e-9
    ]


def target_is_underfilled(region: Region) -> bool:
    return (
        region.contains_island_target
        and region.pop < MIN_POP - 1e-9
    )


def unresolved_target_ids(state: RunState) -> list[int]:
    return [
        rid
        for rid, region in state.regions.items()
        if target_is_underfilled(region) and rid not in state.blocked
    ]


def nearest_distance(state: RunState, rid: int) -> float:
    candidates = nearest_candidates(state, rid, k=1)
    if not candidates:
        return math.inf
    return float(state.distance[rid, candidates[0]])


def target_priority(state: RunState, rid: int, rng: random.Random):
    preferred_count = len(preferred_candidate_ids(state, rid))
    total_count = len(nearest_candidates(state, rid))
    # More remote targets are processed earlier: use negative distance.
    return (
        preferred_count,
        total_count,
        -nearest_distance(state, rid),
        state.regions[rid].pop,
        rng.random(),
        rid,
    )


def candidate_list_after_simulated_merge(
    state: RunState,
    uid: int,
    rid: int,
    sid: int,
    merged_pop: float,
    k: int = K_NEAREST,
) -> tuple[list[int], dict[int, float], dict[int, float]]:
    """
    Return candidate IDs, simulated distances, and simulated populations for uid
    after rid absorbs sid. This does not mutate state.
    """
    ids = active_ids(state, {uid, sid})

    if uid == rid:
        # The merged region's distance to every remaining region is the minimum
        # of target-to-region and candidate-to-region distance.
        dvals = {
            int(v): float(min(state.distance[rid, v], state.distance[sid, v]))
            for v in ids
            if v != rid
        }
        pvals = {int(v): state.regions[int(v)].pop for v in ids if v != rid}
    else:
        dvals = {}
        pvals = {}
        for v0 in ids:
            v = int(v0)
            if v == rid:
                dvals[v] = float(
                    min(state.distance[uid, rid], state.distance[uid, sid])
                )
                pvals[v] = merged_pop
            else:
                dvals[v] = float(state.distance[uid, v])
                pvals[v] = state.regions[v].pop

    ordered = sorted(dvals, key=lambda v: (dvals[v], v))[: min(k, len(dvals))]
    return ordered, dvals, pvals


def simulated_preferred_count(
    state: RunState,
    uid: int,
    rid: int,
    sid: int,
    merged_pop: float,
) -> int:
    candidates, _, pvals = candidate_list_after_simulated_merge(
        state, uid, rid, sid, merged_pop
    )
    uid_pop = merged_pop if uid == rid else state.regions[uid].pop
    return sum(uid_pop + pvals[v] <= SOFT_MAX_POP + 1e-9 for v in candidates)


def stranding_penalty(
    state: RunState,
    rid: int,
    sid: int,
    merged_pop: float,
    before_nearest: dict[int, list[int]],
    before_preferred: dict[int, int],
):
    """Local look-ahead for other unresolved island targets and the merged target."""
    penalty = 0.0
    newly_zero = 0
    newly_one = 0
    affected_count = 0

    for uid in before_nearest:
        if uid in {rid, sid} or not state.active[uid]:
            continue

        # Only targets whose current K-nearest set contains one of the merged
        # regions can lose an existing preferred option.
        if rid not in before_nearest[uid] and sid not in before_nearest[uid]:
            continue

        affected_count += 1
        before = before_preferred[uid]
        after = simulated_preferred_count(state, uid, rid, sid, merged_pop)

        if before > 0 and after == 0:
            penalty += 1.0
            newly_zero += 1
        elif before > 1 and after == 1:
            penalty += 0.25
            newly_one += 1

    self_options = None
    if merged_pop < MIN_POP - 1e-9:
        self_options = simulated_preferred_count(
            state, rid, rid, sid, merged_pop
        )
        if self_options == 0:
            penalty += 1.5
        elif self_options == 1:
            penalty += 0.25

    return penalty, newly_zero, newly_one, affected_count, self_options


def normalized_distance_penalties(
    state: RunState,
    rid: int,
    candidates: list[int],
) -> dict[int, float]:
    if not candidates:
        return {}

    distances = np.array(
        [state.distance[rid, sid] for sid in candidates],
        dtype=float,
    )
    transformed = np.log1p(np.maximum(0.0, distances))
    lo = float(transformed.min())
    hi = float(transformed.max())

    if hi - lo <= DISTANCE_TIE_TOL:
        return {sid: 0.0 for sid in candidates}

    values = (transformed - lo) / (hi - lo)
    return {sid: float(v) for sid, v in zip(candidates, values)}


def evaluate_candidates(
    state: RunState,
    rid: int,
    before_nearest: dict[int, list[int]],
    before_preferred: dict[int, int],
) -> tuple[list[dict], bool, list[int]]:
    r = state.regions[rid]
    nearest = nearest_candidates(state, rid)
    if not nearest:
        return [], False, []

    preferred = [
        sid
        for sid in nearest
        if r.pop + state.regions[sid].pop <= SOFT_MAX_POP + 1e-9
    ]
    candidates = preferred if preferred else nearest
    forced_overflow = not bool(preferred)
    distance_penalties = normalized_distance_penalties(state, rid, candidates)

    evaluated = []
    for sid in candidates:
        s = state.regions[sid]
        merged_pop = r.pop + s.pop
        hetero, da, dc, equivalent, mix = building_penalty(r, s)
        strand, zeroed, one_left, affected, self_options = stranding_penalty(
            state,
            rid,
            sid,
            merged_pop,
            before_nearest,
            before_preferred,
        )
        overflow = max(0.0, merged_pop - SOFT_MAX_POP)
        pop_pen = population_penalty(merged_pop)
        distance_m = float(state.distance[rid, sid])
        distance_pen = distance_penalties[sid]

        score = (
            W_STRAND * strand
            + W_DISTANCE * distance_pen
            + W_HETERO * hetero
            + W_POP * pop_pen
            + W_OVERFLOW * (overflow / MIN_POP)
        )

        evaluated.append(
            {
                "candidate_region": sid,
                "score": score,
                "attachment_distance_m": distance_m,
                "distance_penalty": distance_pen,
                "merged_population": merged_pop,
                "building_penalty": hetero,
                "area_density_rel_diff": da,
                "count_density_rel_diff": dc,
                "building_equivalent_20pct": equivalent,
                "area_balance_mix": mix,
                "stranding_penalty": strand,
                "newly_zero_option_targets": zeroed,
                "newly_one_option_targets": one_left,
                "affected_target_count": affected,
                "merged_self_preferred_options": self_options,
                "population_penalty": pop_pen,
                "overflow_excess": overflow,
                "anchor_target_local_ix": int(state.anchor_left[rid, sid]),
                "anchor_candidate_local_ix": int(state.anchor_right[rid, sid]),
            }
        )

    return evaluated, forced_overflow, nearest


def merge_regions(state: RunState, rid: int, sid: int) -> None:
    """Merge sid into rid and update minimum distances/anchor pairs."""
    r = state.regions[rid]
    s = state.regions[sid]
    active_others = active_ids(state, {rid, sid})

    for k0 in active_others:
        k = int(k0)
        d_r = float(state.distance[rid, k])
        d_s = float(state.distance[sid, k])

        choose_s = d_s < d_r - DISTANCE_TIE_TOL
        if abs(d_s - d_r) <= DISTANCE_TIE_TOL:
            # Deterministic anchor tie-break.
            pair_r = (
                int(state.anchor_left[rid, k]),
                int(state.anchor_right[rid, k]),
            )
            pair_s = (
                int(state.anchor_left[sid, k]),
                int(state.anchor_right[sid, k]),
            )
            choose_s = pair_s < pair_r

        if choose_s:
            new_d = d_s
            left_anchor = int(state.anchor_left[sid, k])
            right_anchor = int(state.anchor_right[sid, k])
        else:
            new_d = d_r
            left_anchor = int(state.anchor_left[rid, k])
            right_anchor = int(state.anchor_right[rid, k])

        state.distance[rid, k] = new_d
        state.distance[k, rid] = new_d
        state.anchor_left[rid, k] = left_anchor
        state.anchor_right[rid, k] = right_anchor
        state.anchor_left[k, rid] = right_anchor
        state.anchor_right[k, rid] = left_anchor

    r.members |= s.members
    r.pop += s.pop
    r.area += s.area
    r.bldg_count += s.bldg_count
    r.bldg_area_sum += s.bldg_area_sum
    r.contains_island_target = (
        r.contains_island_target or s.contains_island_target
    )
    r.initial_target_count += s.initial_target_count
    r.stage2_merge_count += s.stage2_merge_count + 1

    del state.regions[sid]
    state.active[sid] = False
    state.distance[sid, :] = np.inf
    state.distance[:, sid] = np.inf
    state.distance[rid, rid] = 0.0
    state.blocked.discard(rid)
    state.blocked.discard(sid)


def run_once(
    seed: int,
    arrays: tuple[np.ndarray, ...],
    base_distance: np.ndarray,
    base_anchor_left: np.ndarray,
    base_anchor_right: np.ndarray,
):
    rng = random.Random(seed)
    state = make_run_state(
        *arrays,
        base_distance,
        base_anchor_left,
        base_anchor_right,
    )

    merge_rows: list[dict] = []
    audit_rows: list[dict] = []
    iteration = 0

    while True:
        under = unresolved_target_ids(state)
        if not under:
            break

        target = min(target_priority(state, rid, rng) for rid in under)[-1]
        candidates_now = nearest_candidates(state, target)
        if not candidates_now:
            state.blocked.add(target)
            continue

        # Cache current option sets for all unresolved targets once per iteration.
        before_nearest = {
            uid: nearest_candidates(state, uid)
            for uid in under
            if uid != target
        }
        before_preferred = {
            uid: sum(
                state.regions[uid].pop + state.regions[v].pop
                <= SOFT_MAX_POP + 1e-9
                for v in before_nearest[uid]
            )
            for uid in before_nearest
        }

        evaluated, forced_overflow, nearest_all = evaluate_candidates(
            state,
            target,
            before_nearest,
            before_preferred,
        )
        if not evaluated:
            state.blocked.add(target)
            continue

        best_score = min(x["score"] for x in evaluated)
        near_best = [
            x
            for x in evaluated
            if x["score"] <= best_score + NEAR_BEST_BAND
        ]
        chosen = rng.choice(near_best)
        sid = int(chosen["candidate_region"])

        iteration += 1
        target_region = state.regions[target]
        candidate_region = state.regions[sid]

        for item in evaluated:
            audit_row = {
                "iteration": iteration,
                "seed": seed,
                "target_region": target,
                "candidate_region": int(item["candidate_region"]),
                "chosen": int(item is chosen),
                "forced_overflow_candidate_set": int(forced_overflow),
                "target_pop_before": target_region.pop,
                "candidate_pop_before": state.regions[
                    int(item["candidate_region"])
                ].pop,
                "target_stage1_segment_count": len(target_region.members),
                "candidate_stage1_segment_count": len(
                    state.regions[int(item["candidate_region"])].members
                ),
                "target_preferred_option_count": len(
                    preferred_candidate_ids(state, target)
                ),
                "target_nearest_candidate_count": len(nearest_all),
                "near_best_candidate_count": len(near_best),
            }
            audit_row.update(item)
            audit_rows.append(audit_row)

        merge_row = {
            "iteration": iteration,
            "seed": seed,
            "target_region": target,
            "candidate_region": sid,
            "target_pop_before": target_region.pop,
            "candidate_pop_before": candidate_region.pop,
            "target_stage1_segment_count": len(target_region.members),
            "candidate_stage1_segment_count": len(candidate_region.members),
            "target_area_density": area_density(target_region),
            "candidate_area_density": area_density(candidate_region),
            "target_count_density": count_density(target_region),
            "candidate_count_density": count_density(candidate_region),
            "target_preferred_option_count": len(
                preferred_candidate_ids(state, target)
            ),
            "candidate_count_considered": len(evaluated),
            "near_best_candidate_count": len(near_best),
            "forced_overflow": int(forced_overflow),
        }
        merge_row.update(chosen)
        merge_rows.append(merge_row)

        merge_regions(state, target, sid)

    return (
        state,
        pd.DataFrame(merge_rows),
        pd.DataFrame(audit_rows),
    )


def summarize_run(
    seed: int,
    state: RunState,
    merge_log: pd.DataFrame,
) -> dict:
    regions = list(state.regions.values())
    unresolved = np.array(
        [target_is_underfilled(region) for region in regions],
        dtype=bool,
    )
    pops = np.array([region.pop for region in regions], dtype=float)
    over = pops > SOFT_MAX_POP + 1e-9

    if len(merge_log):
        distances = merge_log["attachment_distance_m"].to_numpy(float)
        max_distance = float(distances.max())
        total_distance = float(distances.sum())
        mean_distance = float(distances.mean())
        mean_hetero = float(merge_log["building_penalty"].mean())
    else:
        max_distance = total_distance = mean_distance = mean_hetero = 0.0

    return {
        "seed": seed,
        "final_region_count": len(regions),
        "stage2_merge_count": int(len(merge_log)),
        "unresolved_under500_island_count": int(unresolved.sum()),
        "unresolved_under500_deficit": float(
            sum(
                max(0.0, MIN_POP - region.pop)
                for region in regions
                if target_is_underfilled(region)
            )
        ),
        "over_1000_count": int(over.sum()),
        "over_1000_excess": float(
            np.maximum(0.0, pops - SOFT_MAX_POP).sum()
        ),
        "maximum_population": float(pops.max()),
        "maximum_attachment_distance_m": max_distance,
        "total_attachment_distance_m": total_distance,
        "mean_attachment_distance_m": mean_distance,
        "mean_building_penalty": mean_hetero,
    }


def selection_key(summary: dict):
    return (
        summary["unresolved_under500_island_count"],
        summary["unresolved_under500_deficit"],
        summary["over_1000_excess"],
        summary["over_1000_count"],
        summary["maximum_attachment_distance_m"],
        summary["total_attachment_distance_m"],
        summary["mean_building_penalty"],
        summary["maximum_population"],
    )


# ---------------------------------------------------------------------------
# Geometry helpers for audit lines
# ---------------------------------------------------------------------------


def shortest_line_between(a, b):
    """Shapely 2 shortest_line, with a conservative fallback."""
    try:
        return shapely.shortest_line(a, b)
    except Exception:
        from shapely.geometry import LineString
        from shapely.ops import nearest_points

        p1, p2 = nearest_points(a, b)
        return LineString([p1, p2])


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------


def main() -> None:
    os.makedirs(OUT_FOLDER, exist_ok=True)

    segments_layer = resolve_layer(IN_GPKG, SEGMENTS_LAYER)
    blocks_layer = resolve_layer(IN_GPKG, BLOCKS_LAYER)

    log(f"Reading stage-1 segments: {IN_GPKG} | layer={segments_layer}")
    segments = pyogrio.read_dataframe(
        IN_GPKG,
        layer=segments_layer,
        use_arrow=USE_ARROW,
        force_2d=True,
    ).reset_index(drop=True)

    log(f"Reading stage-1 block crosswalk: layer={blocks_layer}")
    blocks = pyogrio.read_dataframe(
        IN_GPKG,
        layer=blocks_layer,
        use_arrow=USE_ARROW,
        force_2d=True,
    ).reset_index(drop=True)

    if segments.crs is None or not segments.crs.is_projected:
        raise ValueError("Stage-1 segments must use a projected CRS, such as UTM 35S.")
    if blocks.crs != segments.crs:
        raise ValueError("Segments and blocks layers have different CRSs.")

    segment_id_col = resolve_field(segments.columns, SEGMENT_ID)
    zone_col = resolve_field(segments.columns, ZONE_ID)
    zone_pop_col = resolve_field(segments.columns, ZONE_POP)
    type_col = resolve_field(segments.columns, SEGMENT_TYPE)
    component_col = resolve_field(segments.columns, COMPONENT_ID)
    pop_col = resolve_field(segments.columns, POP)
    block_count_col = resolve_field(segments.columns, BLOCK_COUNT)
    area_col = resolve_field(segments.columns, BLOCK_AREA)
    bldg_count_col = resolve_field(segments.columns, BLDG_COUNT)
    bldg_area_sum_col = resolve_field(segments.columns, BLDG_AREA_SUM)

    if segments[segment_id_col].isna().any() or segments[segment_id_col].duplicated().any():
        raise ValueError("Stage-1 segment_id must be non-null and unique.")
    if segments[zone_col].isna().any():
        raise ValueError("zone_id contains null values.")
    if segments.geometry.isna().any() or shapely.is_empty(segments.geometry.to_numpy()).any():
        raise ValueError("Stage-1 segments contain null or empty geometry.")
    if (~shapely.is_valid(segments.geometry.to_numpy())).any():
        raise ValueError("Stage-1 segments contain invalid geometry.")

    segments["stage1_row_ix"] = np.arange(len(segments), dtype=np.int64)
    stage1_ids = segments[segment_id_col].astype(str).to_numpy()
    zone_values = segments[zone_col].to_numpy()
    zone_pop_values_all = pd.to_numeric(
        segments[zone_pop_col], errors="raise"
    ).to_numpy(float)
    segment_types = segments[type_col].astype(str).to_numpy()
    populations = pd.to_numeric(segments[pop_col], errors="raise").to_numpy(float)
    block_counts = pd.to_numeric(
        segments[block_count_col], errors="raise"
    ).to_numpy(float)
    areas = pd.to_numeric(segments[area_col], errors="raise").to_numpy(float)
    bldg_counts = pd.to_numeric(
        segments[bldg_count_col], errors="coerce"
    ).fillna(0.0).to_numpy(float)
    bldg_area_sums = pd.to_numeric(
        segments[bldg_area_sum_col], errors="coerce"
    ).fillna(0.0).to_numpy(float)

    if np.isnan(populations).any() or (populations < 0).any():
        raise ValueError("Segment population must be non-null and >= 0.")
    if np.isnan(areas).any() or (areas <= 0).any():
        raise ValueError("Segment block_area_m2 must be non-null and > 0.")

    target_origin_global = np.array(
        [
            (segment_types[i] in ISLAND_TARGET_TYPES)
            and (populations[i] < MIN_POP - 1e-9)
            and (zone_pop_values_all[i] > SMALL_ZONE_MAX_POP + 1e-9)
            for i in range(len(segments))
        ],
        dtype=bool,
    )

    log(f"Stage-1 segments: {len(segments):,}")
    log(f"Zones: {segments[zone_col].nunique():,}")
    log(f"Initial under-500 island targets: {target_origin_global.sum():,}")

    final_groups: list[dict] = []
    run_summary_rows: list[dict] = []
    best_merge_logs: list[pd.DataFrame] = []
    best_candidate_audits: list[pd.DataFrame] = []
    line_rows: list[dict] = []
    line_geoms: list = []
    zone_rows: list[dict] = []

    zones = sorted(pd.unique(zone_values), key=lambda x: str(x))

    for zone_no, zone in enumerate(zones, start=1):
        zone_global = np.flatnonzero(zone_values == zone)
        zone_pop_unique = pd.Series(zone_pop_values_all[zone_global]).dropna().to_numpy()
        if len(zone_pop_unique) == 0:
            raise ValueError(f"Zone {zone} has no non-null zones_5_pop.")
        if float(zone_pop_unique.max() - zone_pop_unique.min()) > ZONE_POP_CONSISTENCY_TOL:
            raise ValueError(
                f"Zone {zone} has inconsistent zones_5_pop values: "
                f"min={zone_pop_unique.min()}, max={zone_pop_unique.max()}"
            )
        zone_pop_value = float(np.median(zone_pop_unique))

        excluded_global = zone_global[
            np.isin(segment_types[zone_global], list(EXCLUDED_CANDIDATE_TYPES))
        ]
        open_global = zone_global[
            np.isin(segment_types[zone_global], list(OPEN_SEGMENT_TYPES))
        ]
        airport_global = zone_global[
            np.isin(segment_types[zone_global], list(AIRPORT_SEGMENT_TYPES))
        ]
        eligible_global = zone_global[
            ~np.isin(segment_types[zone_global], list(EXCLUDED_CANDIDATE_TYPES))
        ]
        zone_target_global = zone_global[target_origin_global[zone_global]]

        log(
            f"Zone {zone_no:03d}/{len(zones):03d} | ID={zone} | "
            f"stage1_segments={len(zone_global):,} | "
            f"eligible={len(eligible_global):,} | "
            f"merge_excluded={len(excluded_global):,} | "
            f"open_space={len(open_global):,} | "
            f"airport={len(airport_global):,} | "
            f"targets={len(zone_target_global):,} | "
            f"zones_5_pop={zone_pop_value:,.3f}"
        )

        # Open-space and airport-derived stage-1 segments remain one-to-one and unchanged.
        for global_ix in excluded_global:
            final_groups.append(
                {
                    "members": [int(global_ix)],
                    "zone_id": zone,
                    "zones_5_pop": zone_pop_value,
                    "selected_seed": None,
                    "stage2_processed": False,
                }
            )

        # Small zones were already handled as multipart stage-1 segments.
        # Zones with no under-500 island targets also remain unchanged.
        if (
            zone_pop_value <= SMALL_ZONE_MAX_POP + 1e-9
            or len(zone_target_global) == 0
            or len(eligible_global) <= 1
        ):
            for global_ix in eligible_global:
                final_groups.append(
                    {
                        "members": [int(global_ix)],
                        "zone_id": zone,
                        "zones_5_pop": zone_pop_value,
                        "selected_seed": None,
                        "stage2_processed": False,
                    }
                )

            zone_rows.append(
                {
                    "zone_id": zone,
                    "zones_5_pop": zone_pop_value,
                    "stage1_segment_count": len(zone_global),
                    "eligible_non_open_stage1_segments": len(eligible_global),
                    "initial_under500_island_targets": len(zone_target_global),
                    "stage2_run": False,
                    "selected_seed": None,
                    "stage2_merge_count": 0,
                    "unresolved_under500_islands": len(zone_target_global),
                }
            )
            continue

        # ------------------------------------------------------------------
        # Run stage-2 island attachment for this zone.
        # ------------------------------------------------------------------

        zone_seg = segments.iloc[eligible_global].copy().reset_index(drop=True)
        local_to_global = eligible_global.astype(int)
        local_geoms = zone_seg.geometry.to_numpy()
        base_distance = pairwise_distance_matrix(local_geoms)
        base_anchor_left, base_anchor_right = initial_anchor_matrices(len(zone_seg))

        arrays = (
            populations[local_to_global],
            areas[local_to_global],
            bldg_counts[local_to_global],
            bldg_area_sums[local_to_global],
            target_origin_global[local_to_global],
        )

        best = None
        zone_run_rows = []

        for run_no in range(N_RUNS):
            seed = stable_zone_seed(zone, run_no)
            state, merge_log, candidate_audit = run_once(
                seed,
                arrays,
                base_distance,
                base_anchor_left,
                base_anchor_right,
            )
            summary = summarize_run(seed, state, merge_log)
            summary.update(
                {
                    "run_number": run_no + 1,
                    "zone_id": zone,
                    "zones_5_pop": zone_pop_value,
                    "initial_stage1_segment_count": len(zone_seg),
                    "initial_under500_island_targets": int(
                        target_origin_global[local_to_global].sum()
                    ),
                }
            )
            zone_run_rows.append(summary)

            key = selection_key(summary)
            if best is None or key < best[0]:
                best = (
                    key,
                    seed,
                    state,
                    merge_log,
                    candidate_audit,
                )

        assert best is not None
        _, best_seed, best_state, best_log, best_audit = best

        for row in zone_run_rows:
            row["selected"] = int(row["seed"] == best_seed)
            run_summary_rows.append(row)

        # Enrich accepted merge log and full candidate audit with stage-1 IDs.
        if len(best_log):
            best_log = best_log.copy()
            best_log.insert(0, "zone_id", zone)
            best_log.insert(1, "zones_5_pop", zone_pop_value)
            best_log["anchor_target_global_ix"] = best_log[
                "anchor_target_local_ix"
            ].map(lambda x: int(local_to_global[int(x)]))
            best_log["anchor_candidate_global_ix"] = best_log[
                "anchor_candidate_local_ix"
            ].map(lambda x: int(local_to_global[int(x)]))
            best_log["anchor_target_stage1_segment_id"] = best_log[
                "anchor_target_global_ix"
            ].map(lambda x: stage1_ids[int(x)])
            best_log["anchor_candidate_stage1_segment_id"] = best_log[
                "anchor_candidate_global_ix"
            ].map(lambda x: stage1_ids[int(x)])
            best_merge_logs.append(best_log)

            for _, row in best_log.iterrows():
                a_global = int(row["anchor_target_global_ix"])
                b_global = int(row["anchor_candidate_global_ix"])
                line_geom = shortest_line_between(
                    segments.geometry.iloc[a_global],
                    segments.geometry.iloc[b_global],
                )
                line_rows.append(
                    {
                        "zone_id": zone,
                        "seed": best_seed,
                        "iteration": int(row["iteration"]),
                        "target_stage1_segment_id": stage1_ids[a_global],
                        "candidate_stage1_segment_id": stage1_ids[b_global],
                        "attachment_distance_m": float(
                            row["attachment_distance_m"]
                        ),
                        "merged_population": float(row["merged_population"]),
                        "score": float(row["score"]),
                        "building_penalty": float(row["building_penalty"]),
                        "forced_overflow": int(row["forced_overflow"]),
                    }
                )
                line_geoms.append(line_geom)

        if len(best_audit):
            best_audit = best_audit.copy()
            best_audit.insert(0, "zone_id", zone)
            best_audit.insert(1, "zones_5_pop", zone_pop_value)
            best_audit["anchor_target_global_ix"] = best_audit[
                "anchor_target_local_ix"
            ].map(lambda x: int(local_to_global[int(x)]))
            best_audit["anchor_candidate_global_ix"] = best_audit[
                "anchor_candidate_local_ix"
            ].map(lambda x: int(local_to_global[int(x)]))
            best_audit["anchor_target_stage1_segment_id"] = best_audit[
                "anchor_target_global_ix"
            ].map(lambda x: stage1_ids[int(x)])
            best_audit["anchor_candidate_stage1_segment_id"] = best_audit[
                "anchor_candidate_global_ix"
            ].map(lambda x: stage1_ids[int(x)])
            best_candidate_audits.append(best_audit)

        # Convert best current regions back to global stage-1 segment rows.
        for rid in sorted(
            best_state.regions,
            key=lambda x: min(best_state.regions[x].members),
        ):
            member_global = [
                int(local_to_global[local_ix])
                for local_ix in sorted(best_state.regions[rid].members)
            ]
            final_groups.append(
                {
                    "members": member_global,
                    "zone_id": zone,
                    "zones_5_pop": zone_pop_value,
                    "selected_seed": best_seed,
                    "stage2_processed": True,
                }
            )

        selected_summary = next(
            row for row in zone_run_rows if row["seed"] == best_seed
        )
        zone_rows.append(
            {
                "zone_id": zone,
                "zones_5_pop": zone_pop_value,
                "stage1_segment_count": len(zone_global),
                "eligible_non_open_stage1_segments": len(eligible_global),
                "initial_under500_island_targets": len(zone_target_global),
                "stage2_run": True,
                "selected_seed": best_seed,
                "stage2_merge_count": selected_summary["stage2_merge_count"],
                "unresolved_under500_islands": selected_summary[
                    "unresolved_under500_island_count"
                ],
                "unresolved_under500_deficit": selected_summary[
                    "unresolved_under500_deficit"
                ],
                "over_1000_count_after": selected_summary[
                    "over_1000_count"
                ],
                "over_1000_excess_after": selected_summary[
                    "over_1000_excess"
                ],
                "maximum_attachment_distance_m": selected_summary[
                    "maximum_attachment_distance_m"
                ],
                "total_attachment_distance_m": selected_summary[
                    "total_attachment_distance_m"
                ],
            }
        )

    # ----------------------------------------------------------------------
    # Assign stable final IDs and build stage-1 to final crosswalk.
    # ----------------------------------------------------------------------

    final_groups.sort(
        key=lambda x: (
            str(x["zone_id"]),
            min(x["members"]),
        )
    )

    zone_counter: dict[str, int] = {}
    stage1_global_to_final: dict[int, str] = {}
    group_meta_rows: list[dict] = []

    for group in final_groups:
        zone_text = safe_zone_text(group["zone_id"])
        zone_counter[zone_text] = zone_counter.get(zone_text, 0) + 1
        final_id = f"z{zone_text}_f{zone_counter[zone_text]:04d}"

        members = sorted(group["members"])
        source_types = sorted(set(segment_types[members].tolist()))
        source_components = sorted(
            set(segments.iloc[members][component_col].astype(str).tolist())
        )
        source_target_count = int(target_origin_global[members].sum())
        final_pop = float(populations[members].sum())

        source_excluded_types = [
            str(segment_types[ix])
            for ix in members
            if str(segment_types[ix]) in MERGE_EXCLUDED_SEGMENT_TYPES
        ]
        
        if source_excluded_types:
            if len(members) != 1:
                raise RuntimeError(
                    "A merge-excluded stage-1 segment was merged during stage 2. "
                    f"final members={members}, source types={source_excluded_types}"
                )
            stage2_status = source_excluded_types[0]
        
        elif all(segment_types[ix] == "auto_small_zone" for ix in members):
            stage2_status = "auto_small_zone_unchanged"
        
        elif source_target_count > 0:
            if final_pop < MIN_POP - 1e-9:
                stage2_status = "island_under500_unresolved"
            elif final_pop > SOFT_MAX_POP + 1e-9:
                stage2_status = "island_resolved_over1000"
            else:
                stage2_status = "island_resolved"
        elif len(members) > 1:
            # This is unusual but can occur if a target was absorbed into a region
            # whose surviving root did not itself begin as an island target.
            stage2_status = "island_attachment_recipient"
        elif segment_types[members[0]] == "island_deferred" and final_pop >= MIN_POP:
            stage2_status = "island_already_500plus"
        else:
            stage2_status = "unchanged_stage1"

        for global_ix in members:
            if global_ix in stage1_global_to_final:
                raise RuntimeError(
                    f"Stage-1 segment row {global_ix} was assigned more than once."
                )
            stage1_global_to_final[global_ix] = final_id

        group_meta_rows.append(
            {
                "final_segment_id": final_id,
                "zone_id": group["zone_id"],
                "zones_5_pop": group["zones_5_pop"],
                "selected_seed": group["selected_seed"],
                "stage2_processed": int(group["stage2_processed"]),
                "stage2_status": stage2_status,
                "stage1_segment_count": len(members),
                "stage2_merge_count": max(0, len(members) - 1),
                "initial_island_target_count": source_target_count,
                "source_stage1_types": " | ".join(source_types),
                "source_component_count": len(source_components),
                "source_component_ids": " | ".join(source_components),
                "minimum_stage1_row_ix": min(members),
            }
        )

    if len(stage1_global_to_final) != len(segments):
        missing = sorted(
            set(range(len(segments))) - set(stage1_global_to_final)
        )[:20]
        raise RuntimeError(
            f"Not every stage-1 segment was assigned. First missing rows: {missing}"
        )

    meta = pd.DataFrame(group_meta_rows)
    final_lookup = {
        stage1_ids[ix]: final_id
        for ix, final_id in stage1_global_to_final.items()
    }

    crosswalk = pd.DataFrame(
        {
            "stage1_segment_id": stage1_ids,
            "final_segment_id": [
                stage1_global_to_final[i] for i in range(len(segments))
            ],
            "zone_id": zone_values,
            "zones_5_pop": zone_pop_values_all,
            "stage1_segment_type": segment_types,
            "stage1_component_id": segments[component_col].astype(str),
            "stage1_population": populations,
            "was_initial_under500_island_target": target_origin_global.astype(np.int8),
        }
    )

    # ----------------------------------------------------------------------
    # Final segment geometry and additive attributes.
    # ----------------------------------------------------------------------

    stage1_for_dissolve = segments.copy()
    stage1_for_dissolve["final_segment_id"] = [
        stage1_global_to_final[i] for i in range(len(segments))
    ]

    log("Dissolving stage-1 segments to final stage-2 segments...")
    final_geoms = stage1_for_dissolve[
        ["final_segment_id", "geometry"]
    ].dissolve(by="final_segment_id", as_index=False)

    aggregates = pd.DataFrame(
        {
            "final_segment_id": stage1_for_dissolve["final_segment_id"],
            "population": populations,
            "block_count": block_counts,
            "block_area_m2": areas,
            "bldg_count": bldg_counts,
            "bldg_area_sum": bldg_area_sums,
        }
    ).groupby("final_segment_id", as_index=False).agg(
        population=("population", "sum"),
        block_count=("block_count", "sum"),
        block_area_m2=("block_area_m2", "sum"),
        bldg_count=("bldg_count", "sum"),
        bldg_area_sum=("bldg_area_sum", "sum"),
    )

    final_segments = (
        final_geoms.merge(meta, on="final_segment_id", validate="one_to_one")
        .merge(aggregates, on="final_segment_id", validate="one_to_one")
    )
    final_segments["block_count"] = final_segments["block_count"].round().astype("int64")
    final_segments["bldg_count"] = final_segments["bldg_count"].round().astype("int64")
    final_segments["bldg_area_density"] = (
        final_segments["bldg_area_sum"] / final_segments["block_area_m2"]
    )
    final_segments["bldg_count_density"] = final_segments["bldg_count"] / (
        final_segments["block_area_m2"] / 10_000.0
    )
    final_segments["pop_status"] = final_segments["population"].map(population_status)
    final_segments["pop_shortfall_500"] = np.maximum(
        0.0, MIN_POP - final_segments["population"]
    )
    final_segments["pop_excess_1000"] = np.maximum(
        0.0, final_segments["population"] - SOFT_MAX_POP
    )
    final_segments["segment_area_geom_m2"] = final_segments.geometry.area
    final_segments["segment_perimeter_m"] = final_segments.geometry.length
    final_segments["geometry_part_count"] = shapely.get_num_geometries(
        final_segments.geometry.to_numpy()
    )
    final_segments["is_multipart"] = (
        final_segments["geometry_part_count"] > 1
    ).astype(np.int8)

    # Attach per-final-segment distance summaries from chosen merge logs.
    merge_log_best = (
        pd.concat(best_merge_logs, ignore_index=True)
        if best_merge_logs
        else pd.DataFrame()
    )
    candidate_audit_best = (
        pd.concat(best_candidate_audits, ignore_index=True)
        if best_candidate_audits
        else pd.DataFrame()
    )

    if len(merge_log_best):
        merge_log_best["final_segment_id"] = merge_log_best[
            "anchor_target_stage1_segment_id"
        ].map(final_lookup)
        distance_summary = merge_log_best.groupby(
            "final_segment_id", as_index=False
        ).agg(
            attachment_count=("iteration", "size"),
            attachment_distance_sum_m=("attachment_distance_m", "sum"),
            attachment_distance_max_m=("attachment_distance_m", "max"),
            attachment_distance_mean_m=("attachment_distance_m", "mean"),
            mean_stage2_building_penalty=("building_penalty", "mean"),
            forced_overflow_merge_count=("forced_overflow", "sum"),
        )
        final_segments = final_segments.merge(
            distance_summary,
            on="final_segment_id",
            how="left",
            validate="one_to_one",
        )
    else:
        for name in [
            "attachment_count",
            "attachment_distance_sum_m",
            "attachment_distance_max_m",
            "attachment_distance_mean_m",
            "mean_stage2_building_penalty",
            "forced_overflow_merge_count",
        ]:
            final_segments[name] = 0.0

    fill_zero_cols = [
        "attachment_count",
        "attachment_distance_sum_m",
        "attachment_distance_max_m",
        "attachment_distance_mean_m",
        "mean_stage2_building_penalty",
        "forced_overflow_merge_count",
    ]
    for col in fill_zero_cols:
        if col in final_segments:
            final_segments[col] = final_segments[col].fillna(0)
    final_segments["attachment_count"] = final_segments[
        "attachment_count"
    ].astype("int64")
    final_segments["forced_overflow_merge_count"] = final_segments[
        "forced_overflow_merge_count"
    ].astype("int64")

    # ----------------------------------------------------------------------
    # Block-level final crosswalk.
    # ----------------------------------------------------------------------

    blocks_stage1_id_col = resolve_field(blocks.columns, SEGMENT_ID)
    if blocks[blocks_stage1_id_col].isna().any():
        raise ValueError("blocks_with_segment_id contains null stage-1 segment IDs.")

    blocks_out = blocks.copy()
    blocks_out["stage1_segment_id"] = blocks_out[blocks_stage1_id_col].astype(str)
    blocks_out["final_segment_id"] = blocks_out["stage1_segment_id"].map(final_lookup)
    if blocks_out["final_segment_id"].isna().any():
        missing_ids = blocks_out.loc[
            blocks_out["final_segment_id"].isna(), "stage1_segment_id"
        ].unique()[:20]
        raise RuntimeError(
            "Some block stage-1 IDs were not found in the segment crosswalk: "
            f"{missing_ids.tolist()}"
        )

    # Preserve original first-stage metadata, then make segment_id represent final.
    if "segment_type" in blocks_out.columns:
        blocks_out = blocks_out.rename(
            columns={"segment_type": "stage1_segment_type"}
        )
    if "component_id" in blocks_out.columns:
        blocks_out = blocks_out.rename(
            columns={"component_id": "stage1_component_id"}
        )
    if blocks_stage1_id_col != "stage1_segment_id":
        blocks_out = blocks_out.drop(columns=[blocks_stage1_id_col])

    blocks_out["segment_id"] = blocks_out["final_segment_id"]
    final_meta_lookup = final_segments.set_index("final_segment_id")
    blocks_out["segment_type"] = blocks_out["final_segment_id"].map(
        final_meta_lookup["stage2_status"]
    )
    blocks_out["selected_seed_stage2"] = blocks_out[
        "final_segment_id"
    ].map(final_meta_lookup["selected_seed"])

    # ----------------------------------------------------------------------
    # Attachment-line layer and final QA summaries.
    # ----------------------------------------------------------------------

    if line_rows:
        attachment_lines = gpd.GeoDataFrame(
            pd.DataFrame(line_rows),
            geometry=line_geoms,
            crs=segments.crs,
        )
        attachment_lines["final_segment_id"] = attachment_lines[
            "target_stage1_segment_id"
        ].map(final_lookup)
    else:
        attachment_lines = gpd.GeoDataFrame(
            columns=[
                "zone_id",
                "seed",
                "iteration",
                "target_stage1_segment_id",
                "candidate_stage1_segment_id",
                "attachment_distance_m",
                "merged_population",
                "score",
                "building_penalty",
                "forced_overflow",
                "final_segment_id",
                "geometry",
            ],
            geometry="geometry",
            crs=segments.crs,
        )

    if len(candidate_audit_best):
        candidate_audit_best["chosen_final_segment_id"] = candidate_audit_best[
            "anchor_target_stage1_segment_id"
        ].map(final_lookup)

    # Population preservation.
    if not math.isclose(
        float(final_segments["population"].sum()),
        float(populations.sum()),
        abs_tol=1e-6,
        rel_tol=0.0,
    ):
        raise RuntimeError("Population was not preserved during island resolution.")

    zone_stage2 = pd.DataFrame(zone_rows)
    final_zone = final_segments.groupby("zone_id", as_index=False).agg(
        final_segment_count=("final_segment_id", "size"),
        final_population=("population", "sum"),
        under_500_segments=(
            "pop_status", lambda x: int((x == "under_500").sum())
        ),
        over_1000_segments=(
            "pop_status", lambda x: int((x == "over_1000").sum())
        ),
        total_overflow=("pop_excess_1000", "sum"),
        unresolved_island_segments=(
            "stage2_status",
            lambda x: int((x == "island_under500_unresolved").sum()),
        ),
        island_resolved_segments=(
            "stage2_status",
            lambda x: int(
                x.isin(["island_resolved", "island_resolved_over1000"]).sum()
            ),
        ),
        multipart_segments=("is_multipart", "sum"),
    )
    zone_summary = zone_stage2.merge(
        final_zone,
        on="zone_id",
        how="left",
        validate="one_to_one",
    )

    run_summary = pd.DataFrame(run_summary_rows)

    city_summary = pd.DataFrame(
        [
            {
                "stage1_segment_count": len(segments),
                "initial_under500_island_targets": int(
                    target_origin_global.sum()
                ),
                "final_segment_count": len(final_segments),
    
                # Merge-exclusion QA: stage-1 source segments.
                "stage1_open_space_segments": int(
                    np.isin(segment_types, list(OPEN_SEGMENT_TYPES)).sum()
                ),
                "stage1_airport_segments": int(
                    np.isin(segment_types, list(AIRPORT_SEGMENT_TYPES)).sum()
                ),
                "stage1_merge_excluded_segments": int(
                    np.isin(
                        segment_types,
                        list(MERGE_EXCLUDED_SEGMENT_TYPES),
                    ).sum()
                ),
    
                # Merge-exclusion QA: final stage-2 segments.
                "final_open_space_segments": int(
                    final_segments["stage2_status"]
                    .isin(OPEN_SEGMENT_TYPES)
                    .sum()
                ),
                "final_airport_segments": int(
                    final_segments["stage2_status"]
                    .isin(AIRPORT_SEGMENT_TYPES)
                    .sum()
                ),
                "final_merge_excluded_segments": int(
                    final_segments["stage2_status"]
                    .isin(MERGE_EXCLUDED_SEGMENT_TYPES)
                    .sum()
                ),
    
                "stage2_merges": int(len(merge_log_best)),
                "unresolved_under500_island_segments": int(
                    (
                        final_segments["stage2_status"]
                        == "island_under500_unresolved"
                    ).sum()
                ),
                "island_resolved_segments": int(
                    final_segments["stage2_status"].isin(
                        ["island_resolved", "island_resolved_over1000"]
                    ).sum()
                ),
                "final_under500_segments_all_types": int(
                    (final_segments["pop_status"] == "under_500").sum()
                ),
                "final_over1000_segments": int(
                    (final_segments["pop_status"] == "over_1000").sum()
                ),
                "total_overflow": float(
                    final_segments["pop_excess_1000"].sum()
                ),
                "maximum_attachment_distance_m": (
                    float(merge_log_best["attachment_distance_m"].max())
                    if len(merge_log_best)
                    else 0.0
                ),
                "total_attachment_distance_m": (
                    float(merge_log_best["attachment_distance_m"].sum())
                    if len(merge_log_best)
                    else 0.0
                ),
                "runs_per_zone_with_targets": N_RUNS,
                "k_nearest_candidates": K_NEAREST,
            }
        ]
    )

    # ----------------------------------------------------------------------
    # Write output.
    # ----------------------------------------------------------------------

    if os.path.exists(OUT_GPKG):
        os.remove(OUT_GPKG)

    log(f"Writing output GeoPackage: {OUT_GPKG}")
    pyogrio.write_dataframe(
        final_segments,
        OUT_GPKG,
        layer="segments_islands_resolved",
        driver="GPKG",
        use_arrow=USE_ARROW,
        promote_to_multi=True,
    )
    pyogrio.write_dataframe(
        blocks_out,
        OUT_GPKG,
        layer="blocks_with_final_segment_id",
        driver="GPKG",
        use_arrow=USE_ARROW,
        promote_to_multi=True,
    )
    if len(attachment_lines):
        pyogrio.write_dataframe(
            attachment_lines,
            OUT_GPKG,
            layer="island_attachment_lines",
            driver="GPKG",
            use_arrow=USE_ARROW,
        )

    write_table(city_summary, "city_island_summary.csv", "city_island_summary")
    write_table(zone_summary, "zone_island_summary.csv", "zone_island_summary")
    write_table(run_summary, "island_run_summary.csv", "island_run_summary")
    write_table(
        merge_log_best,
        "island_merge_log_best.csv",
        "island_merge_log_best",
    )
    write_table(
        candidate_audit_best,
        "island_candidate_audit_best.csv",
        "island_candidate_audit_best",
    )
    write_table(
        crosswalk,
        "stage1_to_final_crosswalk.csv",
        "stage1_to_final_crosswalk",
    )
    write_table(
        final_segments.drop(columns="geometry"),
        "final_segment_summary.csv",
        "final_segment_summary",
    )

    log("")
    log("Island-resolution stage complete.")
    log(f"Output: {OUT_GPKG}")
    log(f"Stage-1 segments: {len(segments):,}")
    log(f"Initial under-500 island targets: {target_origin_global.sum():,}")
    log(f"Stage-2 accepted merges: {len(merge_log_best):,}")
    log(f"Final segments: {len(final_segments):,}")
    log(
        "Unresolved under-500 island segments: "
        f"{int((final_segments['stage2_status'] == 'island_under500_unresolved').sum()):,}"
    )
    log(
        "Over-1,000 final segments: "
        f"{int((final_segments['pop_status'] == 'over_1000').sum()):,}"
    )


if __name__ == "__main__":
    try:
        main()
    except Exception:
        log("ERROR:")
        log(traceback.format_exc())
        raise
